<a href="https://colab.research.google.com/github/islam37/S-SLTE-Thesis/blob/main/notebooks/10_s_slte_foundation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from pathlib import Path
import json
import hashlib
import pandas as pd
import torch
from google.colab import drive

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

# Load frozen research protocols.
with (BASE / "configs/holdout_partition_manifest_v1.json").open() as f:
    partition_manifest = json.load(f)

with (BASE / "outputs/llm_main_v1/full_annotation_audit_v1.json").open() as f:
    annotation_audit = json.load(f)

assert annotation_audit["status"] == "FULL_CHECKPOINT_AUDIT_PASSED"
assert annotation_audit["total_records"] == 15621
assert annotation_audit["invalid_predictions"] == 0

# Load clean reference partitions and text-only training data.
anchor_df = pd.read_csv(BASE / "data/splits/anchor.csv")
dev_df = pd.read_csv(BASE / "data/splits/development.csv")

source_training = pd.read_csv(
    BASE / "data/splits/noisy_training.csv",
    usecols=["record_id", "text"]
)

llm_annotations = pd.read_csv(
    BASE / "outputs/llm_main_v1/training_annotations_v1.csv"
)

def id_fingerprint(ids):
    payload = "\n".join(
        str(int(x)) for x in sorted(ids)
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def text_fingerprint(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values("record_id").itertuples(index=False):
        digest.update(str(int(row.record_id)).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(row.text).encode("utf-8"))
        digest.update(b"\0")

    return digest.hexdigest()

def verify_partition(frame, partition):
    spec = partition_manifest["partitions"][partition]
    ids = set(map(int, spec["record_ids"]))

    assert len(frame) == spec["count"]
    assert frame["record_id"].is_unique
    assert set(frame["record_id"].astype(int)) == ids
    assert id_fingerprint(ids) == spec["id_sha256"]
    assert text_fingerprint(frame[["record_id", "text"]]) == (
        spec["raw_text_sha256"]
    )

verify_partition(anchor_df, "anchor")
verify_partition(dev_df, "development")

training_ids = set(map(
    int,
    partition_manifest["partitions"]["training"]["record_ids"]
))

training_text = source_training.loc[
    source_training["record_id"].isin(training_ids)
].copy()

verify_partition(training_text, "training")

assert llm_annotations["record_id"].is_unique
assert set(llm_annotations["record_id"].astype(int)) == training_ids

training_df = training_text.merge(
    llm_annotations,
    on="record_id",
    validate="one_to_one"
)

assert len(training_df) == 13621
assert training_df["llm_label"].isin([0, 1, 2, 3]).all()
assert "label" not in training_df.columns

for frame in [anchor_df, dev_df]:
    assert {"record_id", "text", "label"}.issubset(frame.columns)
    assert frame["label"].isin([0, 1, 2, 3]).all()

assert set(anchor_df["record_id"]).isdisjoint(training_ids)
assert set(dev_df["record_id"]).isdisjoint(training_ids)
assert set(anchor_df["record_id"]).isdisjoint(
    set(dev_df["record_id"])
)

assert torch.cuda.is_available(), "Enable T4 GPU."

print("STAGE 11 — FOUNDATION DATA VALIDATION PASSED")
print("Clean anchor:", len(anchor_df))
print("Development:", len(dev_df))
print("Noisy training:", len(training_df))
print("Frozen fingerprints: VERIFIED")
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("GPU:", torch.cuda.get_device_name(0))
print("Embedding generation: NOT STARTED")


Mounted at /content/drive
STAGE 11 — FOUNDATION DATA VALIDATION PASSED
Clean anchor: 2000
Development: 1800
Noisy training: 13621
Frozen fingerprints: VERIFIED
Training reference labels accessed: NO
Holdout reference labels accessed: NO
GPU: Tesla T4
Embedding generation: NOT STARTED


In [ ]:

%pip install -q "sentence-transformers==5.7.0" "transformers==5.20.1"

import json
import importlib.metadata as metadata

import numpy as np
import torch
from huggingface_hub import HfApi
from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
PROTOCOL_PATH = BASE / "configs/s_slte_embedding_protocol_v1.json"

assert torch.cuda.is_available()
assert metadata.version("sentence-transformers") == "5.7.0"
assert metadata.version("transformers") == "5.20.1"

# Reuse the frozen revision if the protocol already exists.
if PROTOCOL_PATH.exists():
    with PROTOCOL_PATH.open(encoding="utf-8") as f:
        protocol = json.load(f)

    assert protocol["model_id"] == MODEL_ID
    MODEL_REVISION = protocol["model_revision_sha"]
else:
    MODEL_REVISION = HfApi().model_info(MODEL_ID).sha

assert len(MODEL_REVISION) == 40

embedding_model = SentenceTransformer(
    MODEL_ID,
    revision=MODEL_REVISION,
    device="cuda",
    trust_remote_code=False
)

embedding_model.eval()

EMBEDDING_DIM = embedding_model.get_sentence_embedding_dimension()
MAX_SEQUENCE_LENGTH = embedding_model.max_seq_length

assert EMBEDDING_DIM == 384
assert MAX_SEQUENCE_LENGTH == 256

# Small inference test before processing the complete dataset.
sample_vectors = embedding_model.encode(
    training_df["text"].head(4).astype(str).tolist(),
    batch_size=4,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

assert sample_vectors.shape == (4, 384)
assert np.isfinite(sample_vectors).all()
assert np.allclose(
    np.linalg.norm(sample_vectors, axis=1),
    1.0,
    atol=1e-5
)

expected_protocol = {
    "protocol_version": "v1",
    "model_id": MODEL_ID,
    "model_revision_sha": MODEL_REVISION,
    "embedding_dimension": 384,
    "max_sequence_length": 256,
    "normalize_embeddings": True,
    "input_text_policy": "ORIGINAL_TEXT_UNMODIFIED",
    "sentence_transformers_version": "5.7.0",
    "transformers_version": "5.20.1"
}

# Create the experiment protocol without overwriting existing data.
if PROTOCOL_PATH.exists():
    assert protocol == expected_protocol
    protocol_status = "EXISTING_VERIFIED"
else:
    with PROTOCOL_PATH.open("x", encoding="utf-8") as f:
        json.dump(expected_protocol, f, indent=2)
    protocol_status = "NEW_SAVED"

print("STAGE 11 — SENTENCE-BERT VALIDATION PASSED")
print("Model:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("GPU:", torch.cuda.get_device_name(0))
print("Embedding dimension:", EMBEDDING_DIM)
print("Maximum sequence length:", MAX_SEQUENCE_LENGTH)
print("Sample embedding shape:", sample_vectors.shape)
print("Normalized embeddings: VERIFIED")
print("Protocol:", protocol_status)
print("Reference labels used: NO")
print("Full embedding generation: NOT STARTED")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_1277/1243744824.py:39: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBEDDING_DIM = embedding_model.get_sentence_embedding_dimension()


STAGE 11 — SENTENCE-BERT VALIDATION PASSED
Model: sentence-transformers/all-MiniLM-L6-v2
Revision: 1110a243fdf4706b3f48f1d95db1a4f5529b4d41
GPU: Tesla T4
Embedding dimension: 384
Maximum sequence length: 256
Sample embedding shape: (4, 384)
Normalized embeddings: VERIFIED
Protocol: NEW_SAVED
Reference labels used: NO
Full embedding generation: NOT STARTED


In [ ]:

import os
import json
import hashlib
from pathlib import Path

import numpy as np
import torch

# Save embeddings under a versioned, private Drive directory.
EMBEDDING_DIR = BASE / "outputs" / "s_slte_embeddings_v1"
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 64

assert embedding_model.device.type == "cuda"
assert EMBEDDING_DIM == 384
assert MAX_SEQUENCE_LENGTH == 256
assert PROTOCOL_PATH.exists()

with PROTOCOL_PATH.open(encoding="utf-8") as f:
    saved_protocol = json.load(f)

assert saved_protocol["model_id"] == MODEL_ID
assert saved_protocol["model_revision_sha"] == MODEL_REVISION
assert saved_protocol["normalize_embeddings"] is True

partitions = {
    "anchor": anchor_df,
    "development": dev_df,
    "training": training_df
}

expected_counts = {
    "anchor": 2000,
    "development": 1800,
    "training": 13621
}

def verify_embedding_file(path, expected_ids):
    with np.load(path, allow_pickle=False) as saved:
        ids = saved["record_ids"]
        vectors = saved["embeddings"]

    assert np.array_equal(ids, expected_ids)
    assert vectors.shape == (len(expected_ids), 384)
    assert vectors.dtype == np.float32
    assert np.isfinite(vectors).all()

    norms = np.linalg.norm(vectors, axis=1)
    assert np.allclose(norms, 1.0, atol=1e-4)

    return vectors.shape

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

artifact_manifest = {
    "protocol_version": "v1",
    "model_id": MODEL_ID,
    "model_revision_sha": MODEL_REVISION,
    "embedding_dimension": EMBEDDING_DIM,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "normalized": True,
    "batch_size": BATCH_SIZE,
    "input_policy": "ORIGINAL_TEXT_UNMODIFIED",
    "partitions": {}
}

for partition, frame in partitions.items():

    data = (
        frame[["record_id", "text"]]
        .sort_values("record_id")
        .reset_index(drop=True)
    )

    assert len(data) == expected_counts[partition]
    assert data["record_id"].is_unique
    assert data["text"].notna().all()

    verify_partition(data, partition)

    ids = data["record_id"].to_numpy(dtype=np.int64)
    path = EMBEDDING_DIR / f"{partition}_embeddings_v1.npz"

    if path.exists():
        status = "EXISTING_VERIFIED"

    else:
        print(f"\nGenerating embeddings: {partition}")

        texts = data["text"].astype(str).tolist()

        vectors = embedding_model.encode(
            texts,
            batch_size=BATCH_SIZE,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        ).astype(np.float32)

        assert vectors.shape == (len(ids), 384)
        assert np.isfinite(vectors).all()
        assert np.allclose(
            np.linalg.norm(vectors, axis=1),
            1.0,
            atol=1e-4
        )

        temporary_path = path.with_suffix(".npz.tmp")

        with temporary_path.open("wb") as f:
            np.savez_compressed(
                f,
                record_ids=ids,
                embeddings=vectors
            )

        os.replace(temporary_path, path)
        status = "NEW_SAVED"

        del vectors

    shape = verify_embedding_file(path, ids)

    artifact_manifest["partitions"][partition] = {
        "count": len(ids),
        "shape": list(shape),
        "filename": path.name,
        "id_sha256": id_fingerprint(ids),
        "file_sha256": sha256_file(path)
    }

    print(
        f"{partition}: {status} | "
        f"Shape: {shape} | "
        f"Integrity: PASSED"
    )

manifest_path = EMBEDDING_DIR / "embedding_manifest_v1.json"

if manifest_path.exists():
    with manifest_path.open(encoding="utf-8") as f:
        existing_manifest = json.load(f)

    assert existing_manifest == artifact_manifest
    manifest_status = "EXISTING_VERIFIED"

else:
    with manifest_path.open("x", encoding="utf-8") as f:
        json.dump(artifact_manifest, f, indent=2)

    manifest_status = "NEW_SAVED"

print("\nSTAGE 11 — EMBEDDING GENERATION SUMMARY")
print("Total:", sum(expected_counts.values()))
print("Dimension:", EMBEDDING_DIM)
print("Partitions:", len(partitions))
print("Model revision: VERIFIED")
print("Normalized vectors: VERIFIED")
print("Frozen partition integrity: VERIFIED")
print("Embedding manifest:", manifest_status)
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("Files:", EMBEDDING_DIR)
print("\nSTATUS: EMBEDDINGS GENERATED AND VERIFIED")



Generating embeddings: anchor


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

anchor: NEW_SAVED | Shape: (2000, 384) | Integrity: PASSED

Generating embeddings: development


Batches:   0%|          | 0/29 [00:00<?, ?it/s]

development: NEW_SAVED | Shape: (1800, 384) | Integrity: PASSED

Generating embeddings: training


Batches:   0%|          | 0/213 [00:00<?, ?it/s]

training: NEW_SAVED | Shape: (13621, 384) | Integrity: PASSED

STAGE 11 — EMBEDDING GENERATION SUMMARY
Total: 17421
Dimension: 384
Partitions: 3
Model revision: VERIFIED
Normalized vectors: VERIFIED
Frozen partition integrity: VERIFIED
Embedding manifest: NEW_SAVED
Training reference labels accessed: NO
Holdout reference labels accessed: NO
Files: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_embeddings_v1

STATUS: EMBEDDINGS GENERATED AND VERIFIED


In [ ]:

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

assert MAX_SEQUENCE_LENGTH == 256
assert EMBEDDING_DIM == 384

TOKEN_BATCH_SIZE = 256
tokenizer = embedding_model.tokenizer

partitions = {
    "anchor": anchor_df,
    "development": dev_df,
    "training": training_df
}

audit_rows = []
partition_results = {}

for partition, frame in partitions.items():
    data = (
        frame[["record_id", "text"]]
        .sort_values("record_id")
        .reset_index(drop=True)
    )

    verify_partition(data, partition)

    texts = data["text"].astype(str).tolist()
    lengths = []

    for start in range(0, len(texts), TOKEN_BATCH_SIZE):
        batch = texts[start:start + TOKEN_BATCH_SIZE]

        encoded = tokenizer(
            batch,
            add_special_tokens=True,
            truncation=False,
            padding=False,
            return_length=True,
            verbose=False
        )

        lengths.extend(encoded["length"])

    lengths = np.asarray(lengths, dtype=np.int32)

    assert len(lengths) == len(data)
    assert np.all(lengths > 0)

    truncated = lengths > MAX_SEQUENCE_LENGTH
    truncated_count = int(truncated.sum())

    result = {
        "partition": partition,
        "total_records": len(data),
        "truncated_records": truncated_count,
        "truncation_rate_percent": round(
            100 * truncated_count / len(data), 4
        ),
        "mean_tokens": round(float(lengths.mean()), 2),
        "median_tokens": float(np.median(lengths)),
        "p95_tokens": float(np.percentile(lengths, 95)),
        "max_tokens": int(lengths.max()),
        "max_sequence_length": MAX_SEQUENCE_LENGTH,
        "record_id_sha256": id_fingerprint(
            data["record_id"].to_numpy()
        )
    }

    partition_results[partition] = result
    audit_rows.append(result)

audit_df = pd.DataFrame(audit_rows)

total_records = sum(
    item["total_records"]
    for item in partition_results.values()
)

total_truncated = sum(
    item["truncated_records"]
    for item in partition_results.values()
)

assert total_records == 17421

audit_report = {
    "protocol_version": "v1",
    "model_id": MODEL_ID,
    "model_revision_sha": MODEL_REVISION,
    "tokenizer_class": type(tokenizer).__name__,
    "token_count_includes_special_tokens": True,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "input_policy": "ORIGINAL_TEXT_UNMODIFIED",
    "total_records": total_records,
    "total_truncated_records": total_truncated,
    "overall_truncation_rate_percent": round(
        100 * total_truncated / total_records, 4
    ),
    "partitions": partition_results,
    "reference_labels_accessed": False
}

audit_path = EMBEDDING_DIR / "truncation_audit_v1.json"
payload = json.dumps(
    audit_report,
    indent=2,
    ensure_ascii=False
) + "\n"

if audit_path.exists():
    assert audit_path.read_text(encoding="utf-8") == payload
    status = "EXISTING_VERIFIED"
else:
    temporary_path = audit_path.with_suffix(".json.tmp")

    with temporary_path.open("w", encoding="utf-8") as f:
        f.write(payload)

    os.replace(temporary_path, audit_path)
    status = "NEW_SAVED"

print("STAGE 11 — TOKEN TRUNCATION AUDIT")
display(audit_df)

print("Total records:", total_records)
print("Records exceeding 256 tokens:", total_truncated)
print(
    "Overall truncation rate:",
    f"{audit_report['overall_truncation_rate_percent']:.2f}%"
)
print("Audit artifact:", status)
print("Saved path:", audit_path)
print("Reference labels accessed: NO")
print("\nSTATUS: TOKEN TRUNCATION AUDIT PASSED")


STAGE 11 — TOKEN TRUNCATION AUDIT


,partition,total_records,truncated_records,truncation_rate_percent,mean_tokens,median_tokens,p95_tokens,max_tokens,max_sequence_length,record_id_sha256
0,anchor,2000,1,0.0500,52.87,50.0,79.00,379,256,ac474666d787836fec02b33510d4ce0b04074297fb1067...
1,development,1800,2,0.1111,53.25,51.0,81.05,348,256,8d847b8c95dbc031d7826deeb0ac7efa77dbb08a5b598d...
2,training,13621,6,0.0440,53.34,51.0,83.00,351,256,ce13626940296ed9158f4dc116580987f404114f66d42f...


Total records: 17421
Records exceeding 256 tokens: 9
Overall truncation rate: 0.05%
Audit artifact: NEW_SAVED
Saved path: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_embeddings_v1/truncation_audit_v1.json
Reference labels accessed: NO

STATUS: TOKEN TRUNCATION AUDIT PASSED


In [ ]:

import os
import json
import hashlib
from pathlib import Path

import numpy as np

K = 20
QUERY_BATCH_SIZE = 256
KNN_DIR = BASE / "outputs" / "s_slte_knn_v1"
KNN_DIR.mkdir(parents=True, exist_ok=True)

# Verify the frozen embedding artifacts before using them.
with (
    EMBEDDING_DIR / "embedding_manifest_v1.json"
).open(encoding="utf-8") as f:
    embedding_manifest = json.load(f)

assert embedding_manifest["model_id"] == MODEL_ID
assert embedding_manifest["model_revision_sha"] == MODEL_REVISION
assert embedding_manifest["embedding_dimension"] == 384
assert embedding_manifest["normalized"] is True

def file_sha256(path):
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

def load_verified_embeddings(partition, expected_frame):
    path = EMBEDDING_DIR / f"{partition}_embeddings_v1.npz"
    spec = embedding_manifest["partitions"][partition]

    assert file_sha256(path) == spec["file_sha256"]

    with np.load(path, allow_pickle=False) as data:
        ids = data["record_ids"].copy()
        vectors = data["embeddings"].copy()

    expected_ids = (
        expected_frame["record_id"]
        .sort_values()
        .to_numpy(dtype=np.int64)
    )

    assert np.array_equal(ids, expected_ids)
    assert vectors.shape == (len(ids), 384)
    assert vectors.dtype == np.float32
    assert np.isfinite(vectors).all()
    assert np.allclose(
        np.linalg.norm(vectors, axis=1), 1, atol=1e-4
    )

    return ids, vectors

anchor_ids, anchor_vectors = load_verified_embeddings(
    "anchor", anchor_df
)

# Trusted anchor labels are allowed as neighborhood evidence.
anchor_sorted = (
    anchor_df.sort_values("record_id").reset_index(drop=True)
)

anchor_labels = anchor_sorted["label"].to_numpy(dtype=np.int64)

assert np.array_equal(
    anchor_sorted["record_id"].to_numpy(dtype=np.int64),
    anchor_ids
)
assert np.isin(anchor_labels, [0, 1, 2, 3]).all()
assert K <= len(anchor_ids)

def validate_knn(path, query_ids):
    with np.load(path, allow_pickle=False) as saved:
        stored_queries = saved["query_record_ids"].copy()
        neighbor_ids = saved["anchor_record_ids"].copy()
        neighbor_labels = saved["anchor_labels"].copy()
        scores = saved["cosine_similarity"].copy()

    expected_shape = (len(query_ids), K)

    assert np.array_equal(stored_queries, query_ids)
    assert neighbor_ids.shape == expected_shape
    assert neighbor_labels.shape == expected_shape
    assert scores.shape == expected_shape

    assert np.isfinite(scores).all()
    assert np.all(scores >= -1.0001)
    assert np.all(scores <= 1.0001)
    assert np.all(np.diff(scores, axis=1) <= 1e-5)

    anchor_position = {
        int(record_id): index
        for index, record_id in enumerate(anchor_ids)
    }

    indices = np.array([
        [anchor_position[int(rid)] for rid in row]
        for row in neighbor_ids
    ], dtype=np.int64)

    assert np.array_equal(
        neighbor_labels, anchor_labels[indices]
    )

    assert all(
        len(set(row.tolist())) == K
        for row in neighbor_ids
    )

    # Independently verify cosine scores for sampled queries.
    sample_indices = np.unique(
        np.linspace(0, len(query_ids) - 1, 10, dtype=int)
    )

    return indices, scores, sample_indices

knn_manifest = {
    "protocol_version": "v1",
    "model_id": MODEL_ID,
    "model_revision_sha": MODEL_REVISION,
    "similarity": "COSINE_NORMALIZED_DOT_PRODUCT",
    "k": K,
    "neighbor_pool": "CLEAN_ANCHOR_ONLY",
    "tie_break": "ASCENDING_ANCHOR_RECORD_ID",
    "selection_policy": "TOP_K_EXACT",
    "reference_labels_policy": (
        "TRUSTED_ANCHOR_LABELS_ONLY"
    ),
    "partitions": {}
}

for partition, frame in [
    ("development", dev_df),
    ("training", training_df)
]:
    query_ids, query_vectors = load_verified_embeddings(
        partition, frame
    )

    assert set(query_ids).isdisjoint(set(anchor_ids))

    output_path = (
        KNN_DIR / f"{partition}_anchor_knn_k{K}_v1.npz"
    )

    if output_path.exists():
        status = "EXISTING_VERIFIED"

    else:
        all_neighbor_ids = []
        all_neighbor_labels = []
        all_scores = []

        for start in range(0, len(query_ids), QUERY_BATCH_SIZE):
            query_batch = query_vectors[
                start:start + QUERY_BATCH_SIZE
            ]

            # All embeddings are L2-normalized.
            similarities = query_batch @ anchor_vectors.T

            # Anchor IDs are sorted: stable sort resolves score ties.
            order = np.argsort(
                -similarities,
                axis=1,
                kind="stable"
            )[:, :K]

            selected_scores = np.take_along_axis(
                similarities, order, axis=1
            )

            all_neighbor_ids.append(anchor_ids[order])
            all_neighbor_labels.append(anchor_labels[order])
            all_scores.append(selected_scores)

        neighbor_ids = np.concatenate(all_neighbor_ids)
        neighbor_labels = np.concatenate(all_neighbor_labels)
        scores = np.concatenate(all_scores).astype(np.float32)

        assert neighbor_ids.shape == (len(query_ids), K)
        assert scores.shape == (len(query_ids), K)

        temporary_path = output_path.with_suffix(".npz.tmp")

        with temporary_path.open("wb") as f:
            np.savez_compressed(
                f,
                query_record_ids=query_ids,
                anchor_record_ids=neighbor_ids,
                anchor_labels=neighbor_labels,
                cosine_similarity=scores
            )

        os.replace(temporary_path, output_path)
        status = "NEW_SAVED"

    indices, scores, sample_indices = validate_knn(
        output_path, query_ids
    )

    computed_scores = np.sum(
        query_vectors[sample_indices, None, :]
        * anchor_vectors[indices[sample_indices]],
        axis=2
    )

    assert np.allclose(
        scores[sample_indices],
        computed_scores,
        atol=1e-5
    )

    knn_manifest["partitions"][partition] = {
        "count": len(query_ids),
        "filename": output_path.name,
        "shape": [len(query_ids), K],
        "query_ids_sha256": id_fingerprint(query_ids),
        "file_sha256": file_sha256(output_path)
    }

    print(
        f"{partition}: {status} | "
        f"Queries: {len(query_ids)} | "
        f"Neighbors per query: {K} | "
        "Integrity: PASSED"
    )

manifest_path = KNN_DIR / "anchor_knn_manifest_v1.json"
manifest_content = json.dumps(knn_manifest, indent=2) + "\n"

if manifest_path.exists():
    assert manifest_path.read_text(encoding="utf-8") == (
        manifest_content
    )
    manifest_status = "EXISTING_VERIFIED"

else:
    temporary = manifest_path.with_suffix(".json.tmp")

    with temporary.open("w", encoding="utf-8") as f:
        f.write(manifest_content)

    os.replace(temporary, manifest_path)
    manifest_status = "NEW_SAVED"

print("\nSTAGE 11 — SEMANTIC KNN SUMMARY")
print("Clean anchor pool:", len(anchor_ids))
print("Development queries:", len(dev_df))
print("Training queries:", len(training_df))
print("Neighbors per query:", K)
print("Similarity: COSINE")
print("Neighbor ranking: EXACT")
print("Anchor labels used: YES — TRUSTED")
print("Development labels used for ranking: NO")
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("Manifest:", manifest_status)
print("Output directory:", KNN_DIR)
print("\nSTATUS: SEMANTIC KNN GENERATED AND VERIFIED")


development: NEW_SAVED | Queries: 1800 | Neighbors per query: 20 | Integrity: PASSED
training: NEW_SAVED | Queries: 13621 | Neighbors per query: 20 | Integrity: PASSED

STAGE 11 — SEMANTIC KNN SUMMARY
Clean anchor pool: 2000
Development queries: 1800
Training queries: 13621
Neighbors per query: 20
Similarity: COSINE
Neighbor ranking: EXACT
Anchor labels used: YES — TRUSTED
Development labels used for ranking: NO
Training reference labels accessed: NO
Holdout reference labels accessed: NO
Manifest: NEW_SAVED
Output directory: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_knn_v1

STATUS: SEMANTIC KNN GENERATED AND VERIFIED


In [ ]:

import os
import json
import hashlib

import numpy as np
import pandas as pd

NUM_CLASSES = 4
K = 20

EVIDENCE_DIR = BASE / "outputs" / "s_slte_evidence_v1"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

with (
    KNN_DIR / "anchor_knn_manifest_v1.json"
).open(encoding="utf-8") as f:
    knn_manifest = json.load(f)

assert knn_manifest["model_id"] == MODEL_ID
assert knn_manifest["model_revision_sha"] == MODEL_REVISION
assert knn_manifest["k"] == K
assert knn_manifest["neighbor_pool"] == "CLEAN_ANCHOR_ONLY"

def compute_neighborhood_evidence(partition, frame):
    knn_path = (
        KNN_DIR / f"{partition}_anchor_knn_k{K}_v1.npz"
    )

    assert file_sha256(knn_path) == (
        knn_manifest["partitions"][partition]["file_sha256"]
    )

    with np.load(knn_path, allow_pickle=False) as data:
        query_ids = data["query_record_ids"].copy()
        neighbor_labels = data["anchor_labels"].copy()
        similarities = data["cosine_similarity"].copy()

    expected_ids = (
        frame["record_id"]
        .sort_values()
        .to_numpy(dtype=np.int64)
    )

    assert np.array_equal(query_ids, expected_ids)
    assert neighbor_labels.shape == (len(query_ids), K)
    assert similarities.shape == (len(query_ids), K)
    assert np.isin(neighbor_labels, range(NUM_CLASSES)).all()
    assert np.isfinite(similarities).all()

    weights = np.maximum(
        similarities.astype(np.float64), 0.0
    )
    total_weights = weights.sum(axis=1)
    has_evidence = total_weights > 0

    weighted_class_totals = np.zeros(
        (len(query_ids), NUM_CLASSES), dtype=np.float64
    )

    vote_counts = np.zeros(
        (len(query_ids), NUM_CLASSES), dtype=np.int32
    )

    for class_id in range(NUM_CLASSES):
        matches = neighbor_labels == class_id

        weighted_class_totals[:, class_id] = (
            weights * matches
        ).sum(axis=1)

        vote_counts[:, class_id] = matches.sum(axis=1)

    probabilities = np.divide(
        weighted_class_totals,
        total_weights[:, None],
        out=np.zeros_like(weighted_class_totals),
        where=has_evidence[:, None]
    )

    assert np.allclose(
        probabilities[has_evidence].sum(axis=1),
        1.0, atol=1e-7
    )
    assert np.all(probabilities >= 0)
    assert np.all(probabilities <= 1.000001)
    assert np.all(vote_counts.sum(axis=1) == K)

    top_label = probabilities.argmax(axis=1)
    top_label = np.where(has_evidence, top_label, -1)

    sorted_probabilities = np.sort(
        probabilities, axis=1
    )

    margin = (
        sorted_probabilities[:, -1]
        - sorted_probabilities[:, -2]
    )

    result = pd.DataFrame({
        "record_id": query_ids,
        "has_positive_similarity": has_evidence,
        "neighbor_top_label": top_label,
        "neighbor_top_support": probabilities.max(axis=1),
        "neighbor_support_margin": margin,
        "mean_cosine_similarity": similarities.mean(axis=1),
        "max_cosine_similarity": similarities.max(axis=1),
        "top_class_vote_count": vote_counts.max(axis=1)
    })

    for class_id in range(NUM_CLASSES):
        result[f"support_class_{class_id}"] = (
            probabilities[:, class_id]
        )

    if partition == "training":
        assert "label" not in frame.columns

        llm_labels = (
            frame.sort_values("record_id")["llm_label"]
            .to_numpy(dtype=np.int64)
        )

        assert np.isin(llm_labels, range(NUM_CLASSES)).all()

        result["llm_label"] = llm_labels
        result["llm_label_support"] = probabilities[
            np.arange(len(query_ids)), llm_labels
        ]

        result["neighbor_llm_agreement"] = (
            (top_label == llm_labels) & has_evidence
        )

    assert result["record_id"].is_unique
    assert len(result) == len(frame)

    return result

summary = {}

for partition, frame in [
    ("development", dev_df),
    ("training", training_df)
]:
    evidence = compute_neighborhood_evidence(
        partition, frame
    )

    output_path = (
        EVIDENCE_DIR / f"{partition}_neighbor_evidence_v1.csv"
    )

    csv_content = evidence.to_csv(
        index=False,
        float_format="%.8f",
        lineterminator="\n"
    )

    if output_path.exists():
        assert output_path.read_text(
            encoding="utf-8"
        ) == csv_content
        status = "EXISTING_VERIFIED"
    else:
        temporary = output_path.with_suffix(".csv.tmp")
        temporary.write_text(csv_content, encoding="utf-8")
        os.replace(temporary, output_path)
        status = "NEW_SAVED"

    summary[partition] = {
        "records": len(evidence),
        "no_positive_similarity": int(
            (~evidence["has_positive_similarity"]).sum()
        ),
        "file": output_path.name,
        "sha256": file_sha256(output_path)
    }

    if partition == "training":
        summary[partition]["neighbor_llm_agreements"] = int(
            evidence["neighbor_llm_agreement"].sum()
        )
        summary[partition]["neighbor_llm_disagreements"] = int(
            (
                evidence["has_positive_similarity"]
                & ~evidence["neighbor_llm_agreement"]
            ).sum()
        )

    print(
        f"{partition}: {status} | "
        f"Records: {len(evidence)} | "
        f"Without positive similarity: "
        f"{summary[partition]['no_positive_similarity']}"
    )

evidence_manifest = {
    "protocol_version": "v1",
    "model_id": MODEL_ID,
    "model_revision_sha": MODEL_REVISION,
    "k": K,
    "weighting": "MAX_COSINE_ZERO",
    "neighbor_pool": "CLEAN_ANCHOR_ONLY",
    "score_type": "UNCALIBRATED_NEIGHBOR_SUPPORT",
    "label_correction_performed": False,
    "training_reference_labels_accessed": False,
    "holdout_reference_labels_accessed": False,
    "partitions": summary
}

manifest_path = EVIDENCE_DIR / "evidence_manifest_v1.json"
manifest_content = json.dumps(
    evidence_manifest, indent=2
) + "\n"

if manifest_path.exists():
    assert manifest_path.read_text(
        encoding="utf-8"
    ) == manifest_content
    manifest_status = "EXISTING_VERIFIED"
else:
    temporary = manifest_path.with_suffix(".json.tmp")
    temporary.write_text(
        manifest_content, encoding="utf-8"
    )
    os.replace(temporary, manifest_path)
    manifest_status = "NEW_SAVED"

print("\nSTAGE 11 — NEIGHBORHOOD EVIDENCE SUMMARY")
print("Development records:", summary["development"]["records"])
print("Training records:", summary["training"]["records"])
print(
    "Neighbor–LLM agreements:",
    summary["training"]["neighbor_llm_agreements"]
)
print(
    "Neighbor–LLM disagreements:",
    summary["training"]["neighbor_llm_disagreements"]
)
print("Evidence manifest:", manifest_status)
print("Correction performed: NO")
print("Reference-label leakage: NONE")
print("\nSTATUS: NEIGHBORHOOD EVIDENCE GENERATED AND VERIFIED")


development: NEW_SAVED | Records: 1800 | Without positive similarity: 0
training: NEW_SAVED | Records: 13621 | Without positive similarity: 0

STAGE 11 — NEIGHBORHOOD EVIDENCE SUMMARY
Development records: 1800
Training records: 13621
Neighbor–LLM agreements: 11031
Neighbor–LLM disagreements: 2590
Evidence manifest: NEW_SAVED
Correction performed: NO
Reference-label leakage: NONE

STATUS: NEIGHBORHOOD EVIDENCE GENERATED AND VERIFIED


In [ ]:

import os
import json
import hashlib

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# Load and independently verify the saved evidence.
EVIDENCE_DIR = BASE / "outputs" / "s_slte_evidence_v1"

with (EVIDENCE_DIR / "evidence_manifest_v1.json").open(
    encoding="utf-8"
) as f:
    evidence_manifest = json.load(f)

assert evidence_manifest["k"] == 20
assert evidence_manifest["neighbor_pool"] == "CLEAN_ANCHOR_ONLY"
assert evidence_manifest["label_correction_performed"] is False

evidence_path = EVIDENCE_DIR / "development_neighbor_evidence_v1.csv"

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

assert sha256_file(evidence_path) == (
    evidence_manifest["partitions"]["development"]["sha256"]
)

evidence = pd.read_csv(evidence_path)

assert len(evidence) == 1800
assert evidence["record_id"].is_unique
assert evidence["has_positive_similarity"].all()

# Clean development labels are permitted for diagnostic evaluation.
dev_labels = dev_df[["record_id", "label"]].copy()

evaluation = evidence.merge(
    dev_labels,
    on="record_id",
    how="inner",
    validate="one_to_one"
).sort_values("record_id").reset_index(drop=True)

assert len(evaluation) == 1800
assert evaluation["label"].isin(range(4)).all()
assert evaluation["neighbor_top_label"].isin(range(4)).all()

support_columns = [
    f"support_class_{class_id}"
    for class_id in range(4)
]

support_matrix = evaluation[support_columns].to_numpy()

assert np.isfinite(support_matrix).all()
assert np.allclose(
    support_matrix.sum(axis=1), 1.0, atol=1e-6
)
assert np.allclose(
    support_matrix.max(axis=1),
    evaluation["neighbor_top_support"],
    atol=1e-6
)

y_true = evaluation["label"].to_numpy(dtype=int)
y_pred = evaluation["neighbor_top_label"].to_numpy(dtype=int)

accuracy = float(accuracy_score(y_true, y_pred))
macro_f1 = float(
    f1_score(y_true, y_pred, labels=[0, 1, 2, 3],
             average="macro", zero_division=0)
)

class_names = ["World", "Sports", "Business", "Sci/Tech"]

class_results = []

for class_id, class_name in enumerate(class_names):
    mask = y_true == class_id

    class_results.append({
        "class_id": class_id,
        "class_name": class_name,
        "records": int(mask.sum()),
        "correct": int((y_pred[mask] == y_true[mask]).sum()),
        "accuracy": round(
            float((y_pred[mask] == y_true[mask]).mean()), 6
        )
    })

# Predefined thresholds for descriptive coverage/accuracy analysis.
threshold_results = []

for threshold in [0.50, 0.60, 0.70, 0.80, 0.90]:
    accepted = (
        evaluation["neighbor_top_support"].to_numpy()
        >= threshold
    )

    count = int(accepted.sum())

    threshold_results.append({
        "minimum_top_support": threshold,
        "selected_records": count,
        "coverage": round(count / len(evaluation), 6),
        "accuracy_if_selected": (
            round(
                float(accuracy_score(
                    y_true[accepted], y_pred[accepted]
                )), 6
            )
            if count else None
        )
    })

confusion = confusion_matrix(
    y_true, y_pred, labels=[0, 1, 2, 3]
)

report = {
    "protocol_version": "v1",
    "evaluation_partition": "development",
    "records": 1800,
    "neighbor_pool": "CLEAN_ANCHOR_ONLY",
    "k": 20,
    "accuracy": round(accuracy, 6),
    "macro_f1": round(macro_f1, 6),
    "class_metrics": class_results,
    "confusion_matrix": confusion.tolist(),
    "confidence_threshold_diagnostics": threshold_results,
    "threshold_selected": False,
    "label_correction_performed": False,
    "training_reference_labels_accessed": False,
    "holdout_reference_labels_accessed": False,
    "source_evidence_sha256": sha256_file(evidence_path)
}

report_path = EVIDENCE_DIR / "development_neighbor_diagnostics_v1.json"
payload = json.dumps(report, indent=2) + "\n"

if report_path.exists():
    assert report_path.read_text(encoding="utf-8") == payload
    report_status = "EXISTING_VERIFIED"
else:
    temporary = report_path.with_suffix(".json.tmp")
    temporary.write_text(payload, encoding="utf-8")
    os.replace(temporary, report_path)
    report_status = "NEW_SAVED"

print("STAGE 11 — DEVELOPMENT NEIGHBOR DIAGNOSTICS")
print("Development records:", len(evaluation))
print(f"Top-1 neighbor accuracy: {accuracy:.4%}")
print(f"Macro F1: {macro_f1:.4f}")

print("\nPER-CLASS ACCURACY")
display(pd.DataFrame(class_results))

print("\nCONFUSION MATRIX — ROWS: TRUE, COLUMNS: PREDICTED")
display(pd.DataFrame(
    confusion,
    index=class_names,
    columns=class_names
))

print("\nSUPPORT-THRESHOLD DIAGNOSTICS")
display(pd.DataFrame(threshold_results))

print("\nReport:", report_status)
print("Saved path:", report_path)
print("Correction performed: NO")
print("Threshold selected: NO")
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("\nSTATUS: DEVELOPMENT DIAGNOSTICS PASSED")


STAGE 11 — DEVELOPMENT NEIGHBOR DIAGNOSTICS
Development records: 1800
Top-1 neighbor accuracy: 88.3333%
Macro F1: 0.8831

PER-CLASS ACCURACY


,class_id,class_name,records,correct,accuracy
0,0,World,450,390,0.866667
1,1,Sports,450,439,0.975556
2,2,Business,450,384,0.853333
3,3,Sci/Tech,450,377,0.837778



CONFUSION MATRIX — ROWS: TRUE, COLUMNS: PREDICTED


,World,Sports,Business,Sci/Tech
World,390,15,32,13
Sports,7,439,3,1
Business,11,3,384,52
Sci/Tech,27,6,40,377



SUPPORT-THRESHOLD DIAGNOSTICS


,minimum_top_support,selected_records,coverage,accuracy_if_selected
0,0.5,1690,0.938889,0.910059
1,0.6,1518,0.843333,0.931489
2,0.7,1337,0.742778,0.949140
3,0.8,1112,0.617778,0.965827
4,0.9,768,0.426667,0.984375



Report: NEW_SAVED
Saved path: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_evidence_v1/development_neighbor_diagnostics_v1.json
Correction performed: NO
Threshold selected: NO
Training reference labels accessed: NO
Holdout reference labels accessed: NO

STATUS: DEVELOPMENT DIAGNOSTICS PASSED


In [ ]:

import os
import json
import hashlib
import numpy as np
import pandas as pd

EVIDENCE_DIR = BASE / "outputs" / "s_slte_evidence_v1"

with (EVIDENCE_DIR / "evidence_manifest_v1.json").open(
    encoding="utf-8"
) as f:
    evidence_manifest = json.load(f)

assert evidence_manifest["k"] == 20
assert evidence_manifest["neighbor_pool"] == "CLEAN_ANCHOR_ONLY"

def load_verified_evidence(partition, expected_frame):
    path = EVIDENCE_DIR / f"{partition}_neighbor_evidence_v1.csv"

    assert sha256_file(path) == (
        evidence_manifest["partitions"][partition]["sha256"]
    )

    data = pd.read_csv(path).sort_values(
        "record_id"
    ).reset_index(drop=True)

    expected_ids = (
        expected_frame["record_id"].sort_values()
        .to_numpy(dtype=np.int64)
    )

    assert np.array_equal(
        data["record_id"].to_numpy(dtype=np.int64),
        expected_ids
    )
    assert data["has_positive_similarity"].all()
    assert data["neighbor_top_label"].isin(range(4)).all()
    assert data["neighbor_top_support"].between(0, 1).all()

    return data, path

dev_evidence, dev_path = load_verified_evidence(
    "development", dev_df
)

training_evidence, training_path = load_verified_evidence(
    "training", training_df
)

dev_evaluation = dev_evidence.merge(
    dev_df[["record_id", "label"]],
    on="record_id",
    validate="one_to_one"
)

training_evaluation = training_evidence.merge(
    training_df[["record_id", "llm_label"]],
    on="record_id",
    validate="one_to_one",
    suffixes=("", "_source")
)

assert np.array_equal(
    training_evaluation["llm_label"],
    training_evaluation["llm_label_source"]
)

dev_correct = (
    dev_evaluation["neighbor_top_label"].to_numpy()
    == dev_evaluation["label"].to_numpy()
)

train_conflict = (
    training_evaluation["neighbor_top_label"].to_numpy()
    != training_evaluation["llm_label"].to_numpy()
)

assert len(dev_correct) == 1800
assert len(train_conflict) == 13621
assert int(dev_correct.sum()) == 1590
assert int(train_conflict.sum()) == 2590

dev_support = dev_evaluation[
    "neighbor_top_support"
].to_numpy()

train_support = training_evaluation[
    "neighbor_top_support"
].to_numpy()

threshold_rows = []

for threshold in [0.50, 0.60, 0.70, 0.80, 0.90]:
    dev_selected = dev_support >= threshold
    train_selected = train_support >= threshold

    dev_count = int(dev_selected.sum())
    dev_errors = int((dev_selected & ~dev_correct).sum())

    train_conflicts = int(
        (train_selected & train_conflict).sum()
    )

    threshold_rows.append({
        "threshold": threshold,
        "development_selected": dev_count,
        "development_incorrect": dev_errors,
        "development_selected_error_rate": (
            round(dev_errors / dev_count, 6)
            if dev_count else None
        ),
        "training_high_support_conflicts": train_conflicts,
        "training_conflict_coverage": round(
            train_conflicts / 2590, 6
        )
    })

threshold_table = pd.DataFrame(threshold_rows)

class_rows = []

for class_id, class_name in enumerate(
    ["World", "Sports", "Business", "Sci/Tech"]
):
    mask = (
        training_evaluation["llm_label"].to_numpy()
        == class_id
    )

    class_rows.append({
        "llm_class": class_name,
        "total": int(mask.sum()),
        "neighbor_conflicts": int(
            (mask & train_conflict).sum()
        ),
        "conflict_rate": round(
            float(train_conflict[mask].mean()), 6
        )
    })

class_table = pd.DataFrame(class_rows)

report = {
    "protocol_version": "v1",
    "analysis": "HIGH_CONFIDENCE_NEIGHBOR_CONFLICT_RISK",
    "k": 20,
    "development_records": 1800,
    "training_records": 13621,
    "development_correct_neighbor_predictions": 1590,
    "training_neighbor_llm_conflicts": 2590,
    "thresholds_exploratory_only": True,
    "threshold_selected": False,
    "correction_performed": False,
    "development_labels_used_for_diagnostics": True,
    "training_reference_labels_accessed": False,
    "holdout_reference_labels_accessed": False,
    "threshold_diagnostics": threshold_rows,
    "conflicts_by_llm_class": class_rows,
    "source_sha256": {
        "development": sha256_file(dev_path),
        "training": sha256_file(training_path)
    }
}

report_path = (
    EVIDENCE_DIR / "high_confidence_conflict_audit_v1.json"
)

payload = json.dumps(report, indent=2) + "\n"

if report_path.exists():
    assert report_path.read_text(
        encoding="utf-8"
    ) == payload
    status = "EXISTING_VERIFIED"
else:
    temporary = report_path.with_suffix(".json.tmp")
    temporary.write_text(payload, encoding="utf-8")
    os.replace(temporary, report_path)
    status = "NEW_SAVED"

print("STAGE 11 — HIGH-CONFIDENCE CONFLICT AUDIT")

print("\nSUPPORT THRESHOLD ANALYSIS")
display(threshold_table)

print("\nTRAINING CONFLICTS BY LLM CLASS")
display(class_table)

print("\nTraining conflicts:", int(train_conflict.sum()))
print("Development errors:", int((~dev_correct).sum()))
print("Report:", status)
print("Threshold selected: NO")
print("Correction performed: NO")
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("\nSTATUS: CONFLICT RISK AUDIT PASSED")


STAGE 11 — HIGH-CONFIDENCE CONFLICT AUDIT

SUPPORT THRESHOLD ANALYSIS


,threshold,development_selected,development_incorrect,development_selected_error_rate,training_high_support_conflicts,training_conflict_coverage
0,0.5,1690,152,0.089941,2057,0.794208
1,0.6,1518,104,0.068511,1519,0.586486
2,0.7,1337,68,0.050860,1097,0.423552
3,0.8,1112,38,0.034173,705,0.272201
4,0.9,768,12,0.015625,293,0.113127



TRAINING CONFLICTS BY LLM CLASS


,llm_class,total,neighbor_conflicts,conflict_rate
0,World,4199,1031,0.245535
1,Sports,3532,170,0.048131
2,Business,4015,1209,0.301121
3,Sci/Tech,1875,180,0.096000



Training conflicts: 2590
Development errors: 210
Report: NEW_SAVED
Threshold selected: NO
Correction performed: NO
Training reference labels accessed: NO
Holdout reference labels accessed: NO

STATUS: CONFLICT RISK AUDIT PASSED


In [ ]:

import os
import json
import numpy as np
import pandas as pd

EVIDENCE_DIR = BASE / "outputs" / "s_slte_evidence_v1"
THRESHOLDS = [0.70, 0.80, 0.90]
CLASS_NAMES = ["World", "Sports", "Business", "Sci/Tech"]

with (EVIDENCE_DIR / "evidence_manifest_v1.json").open(
    encoding="utf-8"
) as f:
    evidence_manifest = json.load(f)

def load_evidence(partition):
    path = EVIDENCE_DIR / f"{partition}_neighbor_evidence_v1.csv"

    assert sha256_file(path) == (
        evidence_manifest["partitions"][partition]["sha256"]
    )

    data = pd.read_csv(path)
    assert data["record_id"].is_unique
    assert data["has_positive_similarity"].all()

    return data

dev_evidence = load_evidence("development")
train_evidence = load_evidence("training")

dev = dev_evidence.merge(
    dev_df[["record_id", "label"]],
    on="record_id",
    validate="one_to_one"
)

train = train_evidence.copy()

assert len(dev) == 1800
assert len(train) == 13621
assert "label" not in train.columns

source_labels = training_df.set_index("record_id")["llm_label"]
assert np.array_equal(
    train["llm_label"].to_numpy(),
    source_labels.loc[train["record_id"]].to_numpy()
)

assert dev["neighbor_top_label"].isin(range(4)).all()
assert train["neighbor_top_label"].isin(range(4)).all()

def wilson_upper_bound(errors, n, z=1.959963984540054):
    if n == 0:
        return None

    p = errors / n
    denominator = 1.0 + z**2 / n

    center = (
        p + z**2 / (2 * n)
    ) / denominator

    radius = (
        z * np.sqrt(
            p * (1 - p) / n + z**2 / (4 * n**2)
        )
    ) / denominator

    return round(float(min(1.0, center + radius)), 6)

dev_prediction = dev["neighbor_top_label"].to_numpy()
dev_truth = dev["label"].to_numpy()
dev_support = dev["neighbor_top_support"].to_numpy()

train_prediction = train["neighbor_top_label"].to_numpy()
train_llm = train["llm_label"].to_numpy()
train_support = train["neighbor_top_support"].to_numpy()

train_conflict = train_prediction != train_llm
assert int(train_conflict.sum()) == 2590

rows = []

for threshold in THRESHOLDS:
    for class_id, class_name in enumerate(CLASS_NAMES):
        selected_dev = (
            (dev_prediction == class_id)
            & (dev_support >= threshold)
        )

        selected_train = (
            (train_prediction == class_id)
            & (train_support >= threshold)
            & train_conflict
        )

        n = int(selected_dev.sum())
        errors = int(
            (selected_dev & (dev_prediction != dev_truth)).sum()
        )

        candidates = int(selected_train.sum())

        rows.append({
            "threshold": threshold,
            "proposed_class": class_name,
            "development_selected": n,
            "development_errors": errors,
            "development_error_rate": (
                round(errors / n, 6) if n else None
            ),
            "wilson_95_upper_error_bound": (
                wilson_upper_bound(errors, n)
            ),
            "training_conflict_candidates": candidates
        })

risk_table = pd.DataFrame(rows)

# Validate reported class totals against overall audit.
for threshold in THRESHOLDS:
    subset = risk_table.loc[
        risk_table["threshold"] == threshold
    ]

    selected_dev = int((dev_support >= threshold).sum())
    selected_train = int(
        ((train_support >= threshold) & train_conflict).sum()
    )

    assert subset["development_selected"].sum() == selected_dev
    assert subset["training_conflict_candidates"].sum() == (
        selected_train
    )

    print(f"\nSUPPORT THRESHOLD >= {threshold:.2f}")
    display(
        subset.drop(columns=["threshold"]).reset_index(drop=True)
    )

report = {
    "protocol_version": "v1",
    "analysis": "CLASS_CONDITIONAL_SELECTIVE_RISK",
    "k": 20,
    "thresholds": THRESHOLDS,
    "confidence_interval": "ONE_SIDED_95_PERCENT_WILSON_UPPER",
    "class_conditional_results": rows,
    "threshold_selected": False,
    "label_correction_performed": False,
    "development_reference_labels_used": True,
    "training_reference_labels_accessed": False,
    "holdout_reference_labels_accessed": False,
    "limitation": (
        "Development neighborhood prediction error is not "
        "the same as expected training label-correction error."
    ),
    "source_sha256": {
        "development": sha256_file(
            EVIDENCE_DIR / "development_neighbor_evidence_v1.csv"
        ),
        "training": sha256_file(
            EVIDENCE_DIR / "training_neighbor_evidence_v1.csv"
        )
    }
}

report_path = (
    EVIDENCE_DIR / "class_conditional_risk_audit_v1.json"
)

payload = json.dumps(report, indent=2) + "\n"

if report_path.exists():
    assert report_path.read_text(
        encoding="utf-8"
    ) == payload
    status = "EXISTING_VERIFIED"
else:
    temporary = report_path.with_suffix(".json.tmp")
    temporary.write_text(payload, encoding="utf-8")
    os.replace(temporary, report_path)
    status = "NEW_SAVED"

print("\nSTAGE 11 — CLASS-CONDITIONAL RISK AUDIT")
print("Development records:", len(dev))
print("Training records:", len(train))
print("Total neighbor–LLM conflicts:", int(train_conflict.sum()))
print("Report:", status)
print("Saved path:", report_path)
print("Threshold selected: NO")
print("Corrections performed: NO")
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("\nSTATUS: CLASS-CONDITIONAL RISK AUDIT PASSED")



SUPPORT THRESHOLD >= 0.70


,proposed_class,development_selected,development_errors,development_error_rate,wilson_95_upper_error_bound,training_conflict_candidates
0,World,329,7,0.021277,0.043260,30
1,Sports,425,12,0.028235,0.048698,66
2,Business,295,28,0.094915,0.133759,184
3,Sci/Tech,288,21,0.072917,0.108893,817



SUPPORT THRESHOLD >= 0.80


,proposed_class,development_selected,development_errors,development_error_rate,wilson_95_upper_error_bound,training_conflict_candidates
0,World,280,4,0.014286,0.036149,10
1,Sports,407,8,0.019656,0.038302,49
2,Business,215,16,0.074419,0.117455,96
3,Sci/Tech,210,10,0.047619,0.085423,550



SUPPORT THRESHOLD >= 0.90


,proposed_class,development_selected,development_errors,development_error_rate,wilson_95_upper_error_bound,training_conflict_candidates
0,World,218,1,0.004587,0.025521,2
1,Sports,333,6,0.018018,0.038746,32
2,Business,107,4,0.037383,0.092200,38
3,Sci/Tech,110,1,0.009091,0.049706,221



STAGE 11 — CLASS-CONDITIONAL RISK AUDIT
Development records: 1800
Training records: 13621
Total neighbor–LLM conflicts: 2590
Report: NEW_SAVED
Saved path: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_evidence_v1/class_conditional_risk_audit_v1.json
Threshold selected: NO
Corrections performed: NO
Training reference labels accessed: NO
Holdout reference labels accessed: NO

STATUS: CLASS-CONDITIONAL RISK AUDIT PASSED


In [ ]:

from pathlib import Path
import json
import hashlib

import pandas as pd
import numpy as np
from google.colab import drive

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

if not (BASE / "configs").exists():
    drive.mount("/content/drive")

assert BASE.exists(), "Research workspace not found."

EVIDENCE_DIR = BASE / "outputs/s_slte_evidence_v1"

with (
    BASE / "outputs/llm_main_v1/full_annotation_audit_v1.json"
).open(encoding="utf-8") as f:
    annotation_audit = json.load(f)

with (
    EVIDENCE_DIR / "evidence_manifest_v1.json"
).open(encoding="utf-8") as f:
    evidence_manifest = json.load(f)

with (
    BASE / "configs/s_slte_embedding_protocol_v1.json"
).open(encoding="utf-8") as f:
    embedding_protocol = json.load(f)

MODEL_ID = embedding_protocol["model_id"]
MODEL_REVISION = embedding_protocol["model_revision_sha"]

def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

assert annotation_audit["status"] == "FULL_CHECKPOINT_AUDIT_PASSED"
assert annotation_audit["total_records"] == 15621
assert evidence_manifest["model_id"] == MODEL_ID
assert evidence_manifest["model_revision_sha"] == MODEL_REVISION

for partition in ["development", "training"]:
    path = EVIDENCE_DIR / f"{partition}_neighbor_evidence_v1.csv"
    assert path.exists()
    assert sha256_file(path) == (
        evidence_manifest["partitions"][partition]["sha256"]
    )

print("STAGE 11 — RUNTIME RECOVERY PASSED")
print("Workspace: VERIFIED")
print("Annotation audit: VERIFIED")
print("Embedding protocol: VERIFIED")
print("Neighborhood evidence: VERIFIED")
print("Model:", MODEL_ID)
print("GPU inference required: NO")
print("Previous research files modified: NO")
print("STATUS: READY TO RERUN STEP 10")


Mounted at /content/drive
STAGE 11 — RUNTIME RECOVERY PASSED
Workspace: VERIFIED
Annotation audit: VERIFIED
Embedding protocol: VERIFIED
Neighborhood evidence: VERIFIED
Model: sentence-transformers/all-MiniLM-L6-v2
GPU inference required: NO
Previous research files modified: NO
STATUS: READY TO RERUN STEP 10


In [ ]:

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

SCORE_DIR = BASE / "outputs" / "s_slte_scores_v1"
SCORE_DIR.mkdir(parents=True, exist_ok=True)

source_path = (
    BASE / "outputs/llm_main_v1/training_annotations_v1.csv"
)
evidence_path = (
    EVIDENCE_DIR / "training_neighbor_evidence_v1.csv"
)

assert sha256_file(source_path) == (
    annotation_audit["exports"]["training"]["sha256"]
)
assert sha256_file(evidence_path) == (
    evidence_manifest["partitions"]["training"]["sha256"]
)

evidence = (
    pd.read_csv(evidence_path)
    .sort_values("record_id")
    .reset_index(drop=True)
)

annotations = (
    pd.read_csv(source_path)
    .sort_values("record_id")
    .reset_index(drop=True)
)

assert len(evidence) == len(annotations) == 13621
assert np.array_equal(
    evidence["record_id"].to_numpy(),
    annotations["record_id"].to_numpy()
)
assert np.array_equal(
    evidence["llm_label"].to_numpy(),
    annotations["llm_label"].to_numpy()
)
assert evidence["has_positive_similarity"].all()

class_columns = [f"support_class_{i}" for i in range(4)]
support = evidence[class_columns].to_numpy(dtype=np.float64)
llm_labels = annotations["llm_label"].to_numpy(dtype=int)

assert np.isfinite(support).all()
assert np.all(support >= 0)
assert np.allclose(support.sum(axis=1), 1.0, atol=1e-6)
assert np.isin(llm_labels, range(4)).all()

row_indices = np.arange(len(evidence))

# Semantic support assigned to the original LLM label.
label_trust = support[row_indices, llm_labels]

# Strongest semantic label supported by clean anchors.
neighbor_labels = support.argmax(axis=1)
neighbor_support = support.max(axis=1)

conflict = neighbor_labels != llm_labels

# Zero for agreement; positive when an alternative label is stronger.
support_gap = np.maximum(
    neighbor_support - label_trust, 0.0
)

assert np.allclose(
    label_trust,
    evidence["llm_label_support"].to_numpy(),
    atol=1e-6
)
assert np.array_equal(
    neighbor_labels,
    evidence["neighbor_top_label"].to_numpy()
)
assert int(conflict.sum()) == 2590
assert np.allclose(support_gap[~conflict], 0.0, atol=1e-6)

scores = pd.DataFrame({
    "record_id": evidence["record_id"].to_numpy(dtype=np.int64),
    "llm_label": llm_labels,
    "semantic_label_trustworthiness": label_trust,
    "neighbor_proposed_label": neighbor_labels,
    "neighbor_top_support": neighbor_support,
    "semantic_support_gap": support_gap,
    "llm_neighbor_conflict": conflict,
    "neighbor_top_margin": evidence[
        "neighbor_support_margin"
    ].to_numpy(),
    "max_cosine_similarity": evidence[
        "max_cosine_similarity"
    ].to_numpy()
})

assert scores["record_id"].is_unique
assert "label" not in scores.columns
assert len(scores) == 13621

# Exploratory candidate counts; no thresholds are selected.
diagnostic_rows = []

for min_support in [0.70, 0.80, 0.90]:
    for min_gap in [0.0, 0.2, 0.4]:
        selected = (
            conflict
            & (neighbor_support >= min_support)
            & (support_gap >= min_gap)
        )

        row = {
            "minimum_neighbor_support": min_support,
            "minimum_support_gap": min_gap,
            "candidate_count": int(selected.sum())
        }

        for class_id, class_name in enumerate(
            ["World", "Sports", "Business", "SciTech"]
        ):
            row[f"proposed_{class_name}"] = int(
                (selected & (neighbor_labels == class_id)).sum()
            )

        diagnostic_rows.append(row)

diagnostics = pd.DataFrame(diagnostic_rows)

expected_base_counts = {
    0.70: 1097,
    0.80: 705,
    0.90: 293
}

for threshold, expected_count in expected_base_counts.items():
    row = diagnostics.loc[
        (diagnostics["minimum_neighbor_support"] == threshold)
        & (diagnostics["minimum_support_gap"] == 0.0)
    ]

    assert len(row) == 1
    assert int(row.iloc[0]["candidate_count"]) == expected_count

output_path = (
    SCORE_DIR / "training_semantic_trust_scores_v1.csv"
)

csv_content = scores.to_csv(
    index=False,
    float_format="%.8f",
    lineterminator="\n"
)

if output_path.exists():
    assert output_path.read_text(
        encoding="utf-8"
    ) == csv_content
    score_status = "EXISTING_VERIFIED"
else:
    temporary = output_path.with_suffix(".csv.tmp")
    temporary.write_text(csv_content, encoding="utf-8")
    os.replace(temporary, output_path)
    score_status = "NEW_SAVED"

score_manifest = {
    "protocol_version": "v1",
    "component": "UNCALIBRATED_SEMANTIC_LABEL_SUPPORT",
    "model_id": MODEL_ID,
    "model_revision_sha": MODEL_REVISION,
    "neighbor_pool": "CLEAN_ANCHOR_ONLY",
    "k": 20,
    "records": 13621,
    "conflicts": 2590,
    "score_definition": (
        "Weighted semantic support of the original LLM label"
    ),
    "gap_definition": (
        "Maximum neighbor class support minus LLM label support"
    ),
    "thresholds_selected": False,
    "correction_performed": False,
    "training_reference_labels_accessed": False,
    "holdout_reference_labels_accessed": False,
    "source_sha256": {
        "llm_annotations": sha256_file(source_path),
        "neighbor_evidence": sha256_file(evidence_path)
    },
    "score_file_sha256": sha256_file(output_path),
    "candidate_diagnostics": diagnostic_rows,
    "statistical_note": (
        "Earlier z=1.95996398454 Wilson upper endpoints "
        "are upper limits of two-sided 95% intervals, "
        "equivalently one-sided 97.5% bounds."
    )
}

manifest_path = SCORE_DIR / "semantic_score_manifest_v1.json"
manifest_content = json.dumps(
    score_manifest, indent=2
) + "\n"

if manifest_path.exists():
    assert manifest_path.read_text(
        encoding="utf-8"
    ) == manifest_content
    manifest_status = "EXISTING_VERIFIED"
else:
    temporary = manifest_path.with_suffix(".json.tmp")
    temporary.write_text(
        manifest_content, encoding="utf-8"
    )
    os.replace(temporary, manifest_path)
    manifest_status = "NEW_SAVED"

print("STAGE 11 — SEMANTIC TRUSTWORTHINESS SCORES")
print("Training records:", len(scores))
print("Neighbor–LLM agreements:", int((~conflict).sum()))
print("Neighbor–LLM conflicts:", int(conflict.sum()))

print("\nEXPLORATORY CONFLICT CANDIDATES")
display(diagnostics)

print("\nSEMANTIC TRUSTWORTHINESS SUMMARY")
display(
    scores.groupby("llm_neighbor_conflict")[
        "semantic_label_trustworthiness"
    ].agg(["count", "mean", "median", "min", "max"])
)

print("\nScores:", score_status)
print("Manifest:", manifest_status)
print("Saved directory:", SCORE_DIR)
print("Threshold selected: NO")
print("Correction performed: NO")
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("\nSTATUS: SEMANTIC TRUSTWORTHINESS SCORES VERIFIED")


STAGE 11 — SEMANTIC TRUSTWORTHINESS SCORES
Training records: 13621
Neighbor–LLM agreements: 11031
Neighbor–LLM conflicts: 2590

EXPLORATORY CONFLICT CANDIDATES


,minimum_neighbor_support,minimum_support_gap,candidate_count,proposed_World,proposed_Sports,proposed_Business,proposed_SciTech
0,0.7,0.0,1097,30,66,184,817
1,0.7,0.2,1097,30,66,184,817
2,0.7,0.4,1097,30,66,184,817
3,0.8,0.0,705,10,49,96,550
4,0.8,0.2,705,10,49,96,550
5,0.8,0.4,705,10,49,96,550
6,0.9,0.0,293,2,32,38,221
7,0.9,0.2,293,2,32,38,221
8,0.9,0.4,293,2,32,38,221



SEMANTIC TRUSTWORTHINESS SUMMARY


,count,mean,median,min,max
llm_neighbor_conflict,,,,,
False,11031,0.848687,0.901822,0.282547,1.000000
True,2590,0.207993,0.196306,0.000000,0.499168



Scores: NEW_SAVED
Manifest: NEW_SAVED
Saved directory: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_scores_v1
Threshold selected: NO
Correction performed: NO
Training reference labels accessed: NO
Holdout reference labels accessed: NO

STATUS: SEMANTIC TRUSTWORTHINESS SCORES VERIFIED


In [ ]:

import json
import hashlib
from pathlib import Path

import pandas as pd

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

with (BASE / "configs/holdout_partition_manifest_v1.json").open(
    encoding="utf-8"
) as f:
    frozen_manifest = json.load(f)

with (BASE / "configs/annotation_protocol_v1.json").open(
    encoding="utf-8"
) as f:
    llm_protocol = json.load(f)

with (BASE / "configs/llm_pilot_manifest_v2.json").open(
    encoding="utf-8"
) as f:
    pilot_manifest = json.load(f)

# Load text only. No development reference labels enter inference.
dev_annotation_inputs = pd.read_csv(
    BASE / "data/splits/development.csv",
    usecols=["record_id", "text"]
).sort_values("record_id").reset_index(drop=True)

spec = frozen_manifest["partitions"]["development"]

def fingerprint_ids(ids):
    return hashlib.sha256(
        "\n".join(
            str(int(x)) for x in sorted(ids)
        ).encode("utf-8")
    ).hexdigest()

def fingerprint_text(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values("record_id").itertuples(index=False):
        digest.update(str(int(row.record_id)).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(row.text).encode("utf-8"))
        digest.update(b"\0")

    return digest.hexdigest()

assert len(dev_annotation_inputs) == spec["count"] == 1800
assert dev_annotation_inputs["record_id"].is_unique
assert dev_annotation_inputs["text"].notna().all()
assert dev_annotation_inputs.columns.tolist() == [
    "record_id", "text"
]
assert fingerprint_ids(
    dev_annotation_inputs["record_id"]
) == spec["id_sha256"]
assert fingerprint_text(
    dev_annotation_inputs
) == spec["raw_text_sha256"]

DEV_LLM_MODEL_ID = pilot_manifest["model_id"]
DEV_LLM_REVISION = pilot_manifest["model_revision_sha"]
DEV_SYSTEM_PROMPT = llm_protocol["system_prompt"]

DEV_PROMPT_SHA256 = hashlib.sha256(
    DEV_SYSTEM_PROMPT.encode("utf-8")
).hexdigest()

assert DEV_LLM_MODEL_ID == "meta-llama/Llama-3.1-8B-Instruct"
assert DEV_LLM_REVISION == (
    "0e9e39f249a16976918f6564b8830bc894c89659"
)
assert DEV_PROMPT_SHA256 == (
    "77a8d266f6f9d3e2dcdbeb906182e0d64f0836190fe84617a638d3ab3c5ed794"
)
assert DEV_PROMPT_SHA256 == pilot_manifest["prompt_sha256"]

DEV_ANNOTATION_DIR = (
    BASE / "outputs/llm_development_v1"
)
DEV_ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

dev_manifest = {
    "experiment_id": "s_slte_development_annotation_v1",
    "purpose": "DEVELOPMENT_CORRECTION_CALIBRATION",
    "partition": "development",
    "partition_manifest_version": "v1",
    "expected_records": 1800,
    "development_id_sha256": spec["id_sha256"],
    "development_text_sha256": spec["raw_text_sha256"],
    "model_id": DEV_LLM_MODEL_ID,
    "model_revision_sha": DEV_LLM_REVISION,
    "prompt_sha256": DEV_PROMPT_SHA256,
    "quantization": "4-bit NF4",
    "text_policy": "ORIGINAL_TEXT_UNMODIFIED",
    "generation": {
        "do_sample": False,
        "max_new_tokens": 32
    },
    "reference_labels_in_inference": False
}

manifest_path = (
    DEV_ANNOTATION_DIR / "development_annotation_manifest_v1.json"
)

if manifest_path.exists():
    with manifest_path.open(encoding="utf-8") as f:
        assert json.load(f) == dev_manifest

    status = "EXISTING_VERIFIED"
else:
    with manifest_path.open("x", encoding="utf-8") as f:
        json.dump(dev_manifest, f, indent=2)

    status = "NEW_SAVED"

print("STAGE 11 — DEVELOPMENT ANNOTATION PREPARATION")
print("Records:", len(dev_annotation_inputs))
print("Frozen IDs: VERIFIED")
print("Original text fingerprint: VERIFIED")
print("Model:", DEV_LLM_MODEL_ID)
print("Model revision:", DEV_LLM_REVISION)
print("Prompt fingerprint: VERIFIED")
print("Manifest:", status)
print("Development reference labels loaded: NO")
print("Training reference labels loaded: NO")
print("Holdout reference labels loaded: NO")
print("New inference performed: NO")
print("\nSTATUS: DEVELOPMENT ANNOTATION READY")


STAGE 11 — DEVELOPMENT ANNOTATION PREPARATION
Records: 1800
Frozen IDs: VERIFIED
Original text fingerprint: VERIFIED
Model: meta-llama/Llama-3.1-8B-Instruct
Model revision: 0e9e39f249a16976918f6564b8830bc894c89659
Prompt fingerprint: VERIFIED
Manifest: NEW_SAVED
Development reference labels loaded: NO
Training reference labels loaded: NO
Holdout reference labels loaded: NO
New inference performed: NO

STATUS: DEVELOPMENT ANNOTATION READY


In [1]:

from pathlib import Path
import json
import hashlib

import pandas as pd
import torch
from google.colab import drive

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
DEV_LLM_DIR = BASE / "outputs/llm_development_v1"

assert BASE.exists()
assert DEV_LLM_DIR.exists()

# Restore frozen research configuration.
with (BASE / "configs/holdout_partition_manifest_v1.json").open(
    encoding="utf-8"
) as f:
    partition_manifest = json.load(f)

with (BASE / "configs/annotation_protocol_v1.json").open(
    encoding="utf-8"
) as f:
    annotation_protocol = json.load(f)

with (BASE / "configs/llm_pilot_manifest_v2.json").open(
    encoding="utf-8"
) as f:
    pilot_manifest = json.load(f)

with (
    DEV_LLM_DIR / "development_annotation_manifest_v1.json"
).open(encoding="utf-8") as f:
    development_manifest = json.load(f)

# Restore the exact pinned annotator configuration.
DEV_MODEL_ID = pilot_manifest["model_id"]
DEV_MODEL_REVISION = pilot_manifest["model_revision_sha"]
DEV_SYSTEM_PROMPT = annotation_protocol["system_prompt"]

DEV_PROMPT_HASH = hashlib.sha256(
    DEV_SYSTEM_PROMPT.encode("utf-8")
).hexdigest()

assert DEV_MODEL_ID == "meta-llama/Llama-3.1-8B-Instruct"
assert DEV_MODEL_REVISION == (
    "0e9e39f249a16976918f6564b8830bc894c89659"
)
assert DEV_PROMPT_HASH == (
    "77a8d266f6f9d3e2dcdbeb906182e0d64f0836190fe84617a638d3ab3c5ed794"
)

assert development_manifest["model_id"] == DEV_MODEL_ID
assert development_manifest["model_revision_sha"] == DEV_MODEL_REVISION
assert development_manifest["prompt_sha256"] == DEV_PROMPT_HASH
assert development_manifest["expected_records"] == 1800
assert development_manifest["generation"] == {
    "do_sample": False,
    "max_new_tokens": 32
}
assert development_manifest["reference_labels_in_inference"] is False

# Read development IDs and original text only.
dev_annotation_inputs = pd.read_csv(
    BASE / "data/splits/development.csv",
    usecols=["record_id", "text"]
).sort_values("record_id").reset_index(drop=True)

def fingerprint_ids(ids):
    payload = "\n".join(
        str(int(value)) for value in sorted(ids)
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def fingerprint_text(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values("record_id").itertuples(index=False):
        digest.update(str(int(row.record_id)).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(row.text).encode("utf-8"))
        digest.update(b"\0")

    return digest.hexdigest()

spec = partition_manifest["partitions"]["development"]

assert len(dev_annotation_inputs) == 1800
assert dev_annotation_inputs["record_id"].is_unique
assert dev_annotation_inputs["text"].notna().all()
assert dev_annotation_inputs.columns.tolist() == [
    "record_id", "text"
]

assert fingerprint_ids(
    dev_annotation_inputs["record_id"]
) == spec["id_sha256"]

assert fingerprint_text(
    dev_annotation_inputs
) == spec["raw_text_sha256"]

assert (
    development_manifest["development_id_sha256"]
    == spec["id_sha256"]
)
assert (
    development_manifest["development_text_sha256"]
    == spec["raw_text_sha256"]
)

assert torch.cuda.is_available(), "Enable T4 GPU in Colab."

print("STAGE 12 — DEVELOPMENT ENVIRONMENT RESTORED")
print("Development records:", len(dev_annotation_inputs))
print("Frozen IDs: VERIFIED")
print("Original text fingerprint: VERIFIED")
print("Model:", DEV_MODEL_ID)
print("Model revision:", DEV_MODEL_REVISION)
print("Prompt fingerprint: VERIFIED")
print("Development manifest: VERIFIED")
print("GPU:", torch.cuda.get_device_name(0))
print("Reference labels loaded: NO")
print("Model loaded: NO")
print("New inference performed: NO")
print("\nSTATUS: READY FOR PINNED LLAMA MODEL LOADING")


Mounted at /content/drive
STAGE 12 — DEVELOPMENT ENVIRONMENT RESTORED
Development records: 1800
Frozen IDs: VERIFIED
Original text fingerprint: VERIFIED
Model: meta-llama/Llama-3.1-8B-Instruct
Model revision: 0e9e39f249a16976918f6564b8830bc894c89659
Prompt fingerprint: VERIFIED
Development manifest: VERIFIED
GPU: Tesla T4
Reference labels loaded: NO
Model loaded: NO
New inference performed: NO

STATUS: READY FOR PINNED LLAMA MODEL LOADING


In [4]:

%pip install -q "transformers==5.20.1" "huggingface_hub==1.33.0" "bitsandbytes==0.50.2" "accelerate"

import gc
import torch
import transformers
import bitsandbytes
import importlib.metadata as metadata

from google.colab import userdata
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# Validate environment and frozen annotation protocol.
assert torch.cuda.is_available(), "Enable T4 GPU."
assert transformers.__version__ == "5.20.1"
assert metadata.version("bitsandbytes") == "0.50.2"

assert len(dev_annotation_inputs) == 1800
assert DEV_MODEL_REVISION == development_manifest["model_revision_sha"]
assert DEV_PROMPT_HASH == development_manifest["prompt_sha256"]

# Retrieve the token securely without printing it.
hf_token = userdata.get("HF_TOKEN")
assert hf_token, "Enable HF_TOKEN in Colab Secrets."

# Release unused embedding model memory.
if "embedding_model" in globals():
    del embedding_model
    gc.collect()
    torch.cuda.empty_cache()

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Reuse the existing model when available.
if "dev_model" not in globals():
    dev_tokenizer = AutoTokenizer.from_pretrained(
        DEV_MODEL_ID,
        revision=DEV_MODEL_REVISION,
        token=hf_token
    )

    dev_model = AutoModelForCausalLM.from_pretrained(
        DEV_MODEL_ID,
        revision=DEV_MODEL_REVISION,
        token=hf_token,
        quantization_config=quantization_config,
        device_map={"": 0},
        dtype=torch.float16,
        low_cpu_mem_usage=True
    )

dev_model.eval()

# Verify model identity, GPU placement and quantization.
assert dev_model.config._name_or_path == DEV_MODEL_ID
assert next(dev_model.parameters()).device.type == "cuda"
assert not dev_model.training

quantized_layers = sum(
    isinstance(module, bitsandbytes.nn.Linear4bit)
    for module in dev_model.modules()
)

assert quantized_layers > 0, "4-bit quantization not detected."
assert "dev_tokenizer" in globals()
assert dev_tokenizer.eos_token_id is not None

print("STAGE 12 — PINNED LLAMA MODEL LOADING PASSED")
print("Model:", DEV_MODEL_ID)
print("Revision:", DEV_MODEL_REVISION)
print("Transformers:", transformers.__version__)
print("BitsAndBytes:", metadata.version("bitsandbytes"))
print("GPU:", torch.cuda.get_device_name(0))
print("Model device:", next(dev_model.parameters()).device)
print("4-bit quantized layers:", quantized_layers)
print(
    "Allocated VRAM:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)
print("Quantization: 4-bit NF4")
print("Model mode: EVALUATION")
print("Development records:", len(dev_annotation_inputs))
print("Reference labels accessed: NO")
print("New inference performed: NO")
print("\nSTATUS: READY FOR DEVELOPMENT SMOKE TEST")


STAGE 12 — PINNED LLAMA MODEL LOADING PASSED
Model: meta-llama/Llama-3.1-8B-Instruct
Revision: 0e9e39f249a16976918f6564b8830bc894c89659
Transformers: 5.20.1
BitsAndBytes: 0.50.2
GPU: Tesla T4
Model device: cuda:0
4-bit quantized layers: 224
Allocated VRAM: 5.32 GB
Quantization: 4-bit NF4
Model mode: EVALUATION
Development records: 1800
Reference labels accessed: NO
New inference performed: NO

STATUS: READY FOR DEVELOPMENT SMOKE TEST


In [3]:

import torch
import bitsandbytes as bnb
import transformers
import importlib.metadata as metadata

assert "dev_model" in globals()
assert "dev_tokenizer" in globals()

# Verify model identity and GPU placement.
assert dev_model.config._name_or_path == DEV_MODEL_ID
assert next(dev_model.parameters()).device.type == "cuda"

# Verify the actual 4-bit quantized layers.
quantized_layers = sum(
    isinstance(module, bnb.nn.Linear4bit)
    for module in dev_model.modules()
)

assert quantized_layers > 0, "4-bit quantization not detected."

# Verify evaluation mode and tokenizer.
dev_model.eval()
assert not dev_model.training
assert dev_tokenizer.eos_token_id is not None

assert transformers.__version__ == "5.20.1"
assert metadata.version("bitsandbytes") == "0.50.2"

print("STAGE 12 — PINNED LLAMA MODEL VALIDATION PASSED")
print("Model:", DEV_MODEL_ID)
print("Revision:", DEV_MODEL_REVISION)
print("GPU:", torch.cuda.get_device_name(0))
print("Model device:", next(dev_model.parameters()).device)
print("4-bit quantized layers:", quantized_layers)
print("Transformers:", transformers.__version__)
print("BitsAndBytes:", metadata.version("bitsandbytes"))
print("Model mode: EVALUATION")
print("Development records:", len(dev_annotation_inputs))
print("Reference labels accessed: NO")
print("New inference performed: NO")
print("\nSTATUS: READY FOR DEVELOPMENT SMOKE TEST")


STAGE 12 — PINNED LLAMA MODEL VALIDATION PASSED
Model: meta-llama/Llama-3.1-8B-Instruct
Revision: 0e9e39f249a16976918f6564b8830bc894c89659
GPU: Tesla T4
Model device: cuda:0
4-bit quantized layers: 224
Transformers: 5.20.1
BitsAndBytes: 0.50.2
Model mode: EVALUATION
Development records: 1800
Reference labels accessed: NO
New inference performed: NO

STATUS: READY FOR DEVELOPMENT SMOKE TEST


In [5]:

import json
import os
import hashlib

import pandas as pd
import torch

SMOKE_PATH = DEV_LLM_DIR / "development_smoke_test_v1.json"
SMOKE_COUNT = 8

assert len(dev_annotation_inputs) == 1800
assert dev_model.config._name_or_path == DEV_MODEL_ID
assert not dev_model.training
assert DEV_PROMPT_HASH == development_manifest["prompt_sha256"]

smoke_df = dev_annotation_inputs.head(
    SMOKE_COUNT
).reset_index(drop=True)

assert len(smoke_df) == SMOKE_COUNT
assert smoke_df["record_id"].is_unique
assert smoke_df.columns.tolist() == ["record_id", "text"]

def parse_dev_annotation(response):
    try:
        parsed = json.loads(response.strip())

        if (
            isinstance(parsed, dict)
            and set(parsed) == {"label"}
            and type(parsed["label"]) is int
            and parsed["label"] in (0, 1, 2, 3)
        ):
            return parsed["label"]

    except (ValueError, TypeError, AttributeError):
        pass

    return None

def dev_text_sha256(text):
    return hashlib.sha256(
        str(text).encode("utf-8")
    ).hexdigest()

def verify_smoke_prediction(row, prediction):
    assert prediction["record_id"] == int(row.record_id)
    assert prediction["text_sha256"] == dev_text_sha256(row.text)
    assert prediction["predicted_label"] == parse_dev_annotation(
        prediction["raw_response"]
    )
    assert prediction["is_valid"] == (
        prediction["predicted_label"] is not None
    )

if SMOKE_PATH.exists():
    with SMOKE_PATH.open(encoding="utf-8") as f:
        checkpoint = json.load(f)

    assert checkpoint["model_id"] == DEV_MODEL_ID
    assert checkpoint["model_revision_sha"] == DEV_MODEL_REVISION
    assert checkpoint["prompt_sha256"] == DEV_PROMPT_HASH
    assert checkpoint["generation"] == {
        "do_sample": False,
        "max_new_tokens": 32
    }

    predictions = checkpoint["predictions"]
    assert len(predictions) == SMOKE_COUNT

    for row, prediction in zip(
        smoke_df.itertuples(index=False),
        predictions
    ):
        verify_smoke_prediction(row, prediction)

    status = "EXISTING_VERIFIED"

else:
    predictions = []

    for row in smoke_df.itertuples(index=False):
        messages = [
            {
                "role": "system",
                "content": DEV_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"Classify this news:\n\n{row.text}"
            }
        ]

        inputs = dev_tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True
        ).to("cuda")

        with torch.inference_mode():
            outputs = dev_model.generate(
                **inputs,
                max_new_tokens=32,
                do_sample=False,
                pad_token_id=dev_tokenizer.eos_token_id
            )

        generated = outputs[0][
            inputs["input_ids"].shape[-1]:
        ]

        response = dev_tokenizer.decode(
            generated,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        ).strip()

        parsed_label = parse_dev_annotation(response)

        prediction = {
            "record_id": int(row.record_id),
            "text_sha256": dev_text_sha256(row.text),
            "raw_response": response,
            "predicted_label": parsed_label,
            "is_valid": parsed_label is not None
        }

        verify_smoke_prediction(row, prediction)
        predictions.append(prediction)

        print(
            f"Processed: {len(predictions)}/{SMOKE_COUNT} | "
            f"Valid: {prediction['is_valid']}"
        )

        del inputs, outputs, generated

    checkpoint = {
        "experiment_id": "s_slte_development_annotation_v1",
        "stage": "SMOKE_TEST",
        "model_id": DEV_MODEL_ID,
        "model_revision_sha": DEV_MODEL_REVISION,
        "prompt_sha256": DEV_PROMPT_HASH,
        "generation": {
            "do_sample": False,
            "max_new_tokens": 32
        },
        "predictions": predictions
    }

    temporary = SMOKE_PATH.with_suffix(".json.tmp")

    with temporary.open("w", encoding="utf-8") as f:
        json.dump(checkpoint, f, indent=2)

    os.replace(temporary, SMOKE_PATH)
    status = "NEW_SAVED"

valid_count = sum(
    prediction["is_valid"]
    for prediction in predictions
)

assert valid_count == SMOKE_COUNT, (
    f"Invalid predictions: {SMOKE_COUNT - valid_count}"
)

result_df = pd.DataFrame([
    {
        "record_id": item["record_id"],
        "predicted_label": item["predicted_label"],
        "is_valid": item["is_valid"]
    }
    for item in predictions
])

print("\nSTAGE 12 — DEVELOPMENT SMOKE TEST")
display(result_df)

print("Total records:", SMOKE_COUNT)
print("Valid predictions:", valid_count)
print("Invalid predictions:", SMOKE_COUNT - valid_count)
print("Checkpoint:", status)
print("Saved path:", SMOKE_PATH)
print("Model revision: VERIFIED")
print("Prompt fingerprint: VERIFIED")
print("Reference labels accessed: NO")
print("Full development annotation started: NO")
print("\nSTATUS: DEVELOPMENT SMOKE TEST PASSED")


Processed: 1/8 | Valid: True
Processed: 2/8 | Valid: True
Processed: 3/8 | Valid: True
Processed: 4/8 | Valid: True
Processed: 5/8 | Valid: True
Processed: 6/8 | Valid: True
Processed: 7/8 | Valid: True
Processed: 8/8 | Valid: True

STAGE 12 — DEVELOPMENT SMOKE TEST


,record_id,predicted_label,is_valid
0,38,3,True
1,49,2,True
2,62,0,True
3,78,1,True
4,94,3,True
5,129,3,True
6,143,3,True
7,158,0,True


Total records: 8
Valid predictions: 8
Invalid predictions: 0
Checkpoint: NEW_SAVED
Saved path: /content/drive/MyDrive/S_SLTE_Thesis/outputs/llm_development_v1/development_smoke_test_v1.json
Model revision: VERIFIED
Prompt fingerprint: VERIFIED
Reference labels accessed: NO
Full development annotation started: NO

STATUS: DEVELOPMENT SMOKE TEST PASSED


In [9]:

import os
import json
import time
import hashlib
from pathlib import Path

import pandas as pd
import torch

MAX_NEW_PER_RUN = 500

RECORD_DIR = DEV_LLM_DIR / "records"
RECORD_DIR.mkdir(parents=True, exist_ok=True)

assert len(dev_annotation_inputs) == 1800
assert dev_annotation_inputs["record_id"].is_unique
assert not dev_model.training
assert development_manifest["model_revision_sha"] == DEV_MODEL_REVISION
assert development_manifest["prompt_sha256"] == DEV_PROMPT_HASH

expected_ids = set(
    dev_annotation_inputs["record_id"].astype(int)
)

existing_paths = list(RECORD_DIR.glob("record_*.json"))
existing_ids = {
    int(path.stem.split("_")[-1])
    for path in existing_paths
}

assert len(existing_ids) == len(existing_paths)
assert existing_ids.issubset(expected_ids)

def verify_dev_record(row, result):
    assert result["record_id"] == int(row.record_id)
    assert result["text_sha256"] == dev_text_sha256(row.text)
    assert result["model_id"] == DEV_MODEL_ID
    assert result["model_revision_sha"] == DEV_MODEL_REVISION
    assert result["prompt_sha256"] == DEV_PROMPT_HASH
    assert result["generation"] == {
        "do_sample": False,
        "max_new_tokens": 32
    }

    parsed = parse_dev_annotation(result["raw_response"])

    assert parsed is not None
    assert type(result["predicted_label"]) is int
    assert result["predicted_label"] == parsed
    assert result["is_valid"] is True

def create_dev_prediction(row):
    messages = [
        {
            "role": "system",
            "content": DEV_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"Classify this news:\n\n{row.text}"
        }
    ]

    inputs = dev_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to("cuda")

    with torch.inference_mode():
        outputs = dev_model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=dev_tokenizer.eos_token_id
        )

    generated = outputs[0][
        inputs["input_ids"].shape[-1]:
    ]

    response = dev_tokenizer.decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    ).strip()

    label = parse_dev_annotation(response)

    result = {
        "record_id": int(row.record_id),
        "text_sha256": dev_text_sha256(row.text),
        "model_id": DEV_MODEL_ID,
        "model_revision_sha": DEV_MODEL_REVISION,
        "prompt_sha256": DEV_PROMPT_HASH,
        "generation": {
            "do_sample": False,
            "max_new_tokens": 32
        },
        "raw_response": response,
        "predicted_label": label,
        "is_valid": label is not None,
        "source": "development_inference"
    }

    del inputs, outputs, generated

    return result

# Load and verify the completed smoke test.
with SMOKE_PATH.open(encoding="utf-8") as f:
    smoke_checkpoint = json.load(f)

assert smoke_checkpoint["model_id"] == DEV_MODEL_ID
assert smoke_checkpoint["model_revision_sha"] == DEV_MODEL_REVISION
assert smoke_checkpoint["prompt_sha256"] == DEV_PROMPT_HASH
assert smoke_checkpoint["generation"] == {
    "do_sample": False,
    "max_new_tokens": 32
}

smoke_predictions = {
    item["record_id"]: item
    for item in smoke_checkpoint["predictions"]
}

assert len(smoke_predictions) == 8

new_count = 0
smoke_reused = 0
existing_verified = 0
start_time = time.perf_counter()

for row in dev_annotation_inputs.itertuples(index=False):
    record_id = int(row.record_id)
    path = RECORD_DIR / f"record_{record_id}.json"

    if path.exists():
        with path.open(encoding="utf-8") as f:
            result = json.load(f)

        verify_dev_record(row, result)
        existing_verified += 1
        continue

    if record_id in smoke_predictions:
        smoke = smoke_predictions[record_id]

        assert smoke["record_id"] == record_id
        assert smoke["text_sha256"] == dev_text_sha256(row.text)
        assert smoke["is_valid"] is True

        result = {
            **smoke,
            "model_id": DEV_MODEL_ID,
            "model_revision_sha": DEV_MODEL_REVISION,
            "prompt_sha256": DEV_PROMPT_HASH,
            "generation": {
                "do_sample": False,
                "max_new_tokens": 32
            },
            "source": "verified_smoke_reuse"
        }

        smoke_reused += 1

    else:
        if new_count >= MAX_NEW_PER_RUN:
            continue

        result = create_dev_prediction(row)
        new_count += 1

        if new_count % 100 == 0:
            print(
                f"New predictions: {new_count}/{MAX_NEW_PER_RUN}"
                f" | Current record ID: {record_id}"
            )

    verify_dev_record(row, result)

    temporary = path.with_suffix(".json.tmp")

    with temporary.open("w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)

    os.replace(temporary, path)

saved_paths = list(RECORD_DIR.glob("record_*.json"))

saved_count = len(saved_paths)
remaining = 1800 - saved_count

assert saved_count <= 1800
assert remaining >= 0
assert {
    int(path.stem.split("_")[-1])
    for path in saved_paths
}.issubset(expected_ids)

duration = time.perf_counter() - start_time

print("\nSTAGE 12 — DEVELOPMENT PRODUCTION PROGRESS")
print("Expected records:", 1800)
print("Saved checkpoints:", saved_count)
print("Remaining records:", remaining)
print(f"Completion: {100 * saved_count / 1800:.2f}%")

print("\nCURRENT EXECUTION")
print("New predictions:", new_count)
print("Smoke predictions reused:", smoke_reused)
print("Existing records verified:", existing_verified)
print("Execution duration (seconds):", round(duration, 2))

print("\nINTEGRITY")
print("Frozen development manifest: VERIFIED")
print("Model revision: VERIFIED")
print("Prompt fingerprint: VERIFIED")
print("Reference labels accessed: NO")
print("Checkpoint directory:", RECORD_DIR)

if remaining == 0:
    print("\nSTATUS: ALL DEVELOPMENT RECORDS CHECKPOINTED")
else:
    print("\nSTATUS: CHECKPOINTED — RERUN SAME CELL TO CONTINUE")


New predictions: 100/500 | Current record ID: 106071
New predictions: 200/500 | Current record ID: 113782

STAGE 12 — DEVELOPMENT PRODUCTION PROGRESS
Expected records: 1800
Saved checkpoints: 1800
Remaining records: 0
Completion: 100.00%

CURRENT EXECUTION
New predictions: 292
Smoke predictions reused: 0
Existing records verified: 1508
Execution duration (seconds): 229.23

INTEGRITY
Frozen development manifest: VERIFIED
Model revision: VERIFIED
Prompt fingerprint: VERIFIED
Reference labels accessed: NO
Checkpoint directory: /content/drive/MyDrive/S_SLTE_Thesis/outputs/llm_development_v1/records

STATUS: ALL DEVELOPMENT RECORDS CHECKPOINTED


In [10]:

import os
import json
import hashlib
from collections import Counter
from pathlib import Path

import pandas as pd

# Independent audit of all frozen development annotations.
AUDIT_DIR = BASE / "outputs/llm_development_v1"
RECORD_DIR = AUDIT_DIR / "records"

with (
    AUDIT_DIR / "development_annotation_manifest_v1.json"
).open(encoding="utf-8") as f:
    audit_manifest = json.load(f)

with (
    AUDIT_DIR / "development_smoke_test_v1.json"
).open(encoding="utf-8") as f:
    smoke_checkpoint = json.load(f)

assert audit_manifest["expected_records"] == 1800
assert audit_manifest["model_id"] == DEV_MODEL_ID
assert audit_manifest["model_revision_sha"] == DEV_MODEL_REVISION
assert audit_manifest["prompt_sha256"] == DEV_PROMPT_HASH
assert audit_manifest["reference_labels_in_inference"] is False
assert audit_manifest["generation"] == {
    "do_sample": False,
    "max_new_tokens": 32
}

assert len(dev_annotation_inputs) == 1800
assert dev_annotation_inputs["record_id"].is_unique
assert dev_annotation_inputs.columns.tolist() == [
    "record_id", "text"
]

def file_hash(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

def strict_parse(raw_response):
    try:
        parsed = json.loads(raw_response.strip())
    except (ValueError, TypeError, AttributeError):
        return None

    if (
        isinstance(parsed, dict)
        and set(parsed) == {"label"}
        and type(parsed["label"]) is int
        and parsed["label"] in (0, 1, 2, 3)
    ):
        return parsed["label"]

    return None

def save_immutable(path, content):
    payload = content.encode("utf-8")

    if path.exists():
        assert path.read_bytes() == payload, (
            f"Existing artifact differs: {path}"
        )
        return "EXISTING_VERIFIED"

    temporary = path.with_suffix(path.suffix + ".tmp")

    with temporary.open("wb") as f:
        f.write(payload)

    os.replace(temporary, path)
    assert path.read_bytes() == payload

    return "NEW_SAVED"

expected_ids = set(
    dev_annotation_inputs["record_id"].astype(int)
)

expected_files = {
    f"record_{record_id}.json"
    for record_id in expected_ids
}

actual_files = {
    path.name
    for path in RECORD_DIR.glob("record_*.json")
}

assert actual_files == expected_files, (
    f"Missing: {len(expected_files - actual_files)}; "
    f"Unexpected: {len(actual_files - expected_files)}"
)

assert len(smoke_checkpoint["predictions"]) == 8
assert smoke_checkpoint["model_id"] == DEV_MODEL_ID
assert smoke_checkpoint["model_revision_sha"] == DEV_MODEL_REVISION
assert smoke_checkpoint["prompt_sha256"] == DEV_PROMPT_HASH

smoke_by_id = {
    int(item["record_id"]): item
    for item in smoke_checkpoint["predictions"]
}

assert len(smoke_by_id) == 8
assert set(smoke_by_id) == set(
    dev_annotation_inputs.head(8)["record_id"].astype(int)
)

audited = []
source_counts = Counter()

for index, row in enumerate(
    dev_annotation_inputs.itertuples(index=False),
    start=1
):
    record_id = int(row.record_id)
    path = RECORD_DIR / f"record_{record_id}.json"

    with path.open(encoding="utf-8") as f:
        result = json.load(f)

    original_text_hash = hashlib.sha256(
        str(row.text).encode("utf-8")
    ).hexdigest()

    parsed_label = strict_parse(result["raw_response"])

    assert result["record_id"] == record_id
    assert result["text_sha256"] == original_text_hash
    assert result["model_id"] == DEV_MODEL_ID
    assert result["model_revision_sha"] == DEV_MODEL_REVISION
    assert result["prompt_sha256"] == DEV_PROMPT_HASH
    assert result["generation"] == {
        "do_sample": False,
        "max_new_tokens": 32
    }
    assert parsed_label is not None
    assert type(result["predicted_label"]) is int
    assert result["predicted_label"] == parsed_label
    assert result["is_valid"] is True

    assert result["source"] in (
        "development_inference",
        "verified_smoke_reuse"
    )

    if record_id in smoke_by_id:
        smoke = smoke_by_id[record_id]

        assert result["source"] == "verified_smoke_reuse"
        assert result["raw_response"] == smoke["raw_response"]
        assert result["predicted_label"] == smoke["predicted_label"]
        assert result["text_sha256"] == smoke["text_sha256"]
    else:
        assert result["source"] == "development_inference"

    source_counts[result["source"]] += 1

    audited.append({
        "record_id": record_id,
        "llm_label": parsed_label
    })

    if index % 300 == 0:
        print(f"Audited: {index}/1800")

assert len(audited) == 1800
assert source_counts["verified_smoke_reuse"] == 8
assert source_counts["development_inference"] == 1792

annotation_df = (
    pd.DataFrame(audited)
    .sort_values("record_id")
    .reset_index(drop=True)
)

assert annotation_df["record_id"].is_unique
assert set(annotation_df["record_id"].astype(int)) == expected_ids
assert annotation_df["llm_label"].isin(range(4)).all()

# Export predictions only; reference labels remain separate.
csv_path = AUDIT_DIR / "development_annotations_v1.csv"

csv_content = annotation_df.to_csv(
    index=False,
    lineterminator="\n"
)

csv_status = save_immutable(csv_path, csv_content)

label_distribution = {
    str(class_id): int(
        (annotation_df["llm_label"] == class_id).sum()
    )
    for class_id in range(4)
}

audit_report = {
    "experiment_id": "s_slte_development_annotation_v1",
    "status": "FULL_DEVELOPMENT_ANNOTATION_AUDIT_PASSED",
    "expected_records": 1800,
    "verified_records": len(audited),
    "missing_records": 0,
    "invalid_predictions": 0,
    "smoke_predictions_reused": 8,
    "new_development_predictions": 1792,
    "model_id": DEV_MODEL_ID,
    "model_revision_sha": DEV_MODEL_REVISION,
    "prompt_sha256": DEV_PROMPT_HASH,
    "generation": {
        "do_sample": False,
        "max_new_tokens": 32
    },
    "development_id_sha256": (
        audit_manifest["development_id_sha256"]
    ),
    "development_text_sha256": (
        audit_manifest["development_text_sha256"]
    ),
    "label_distribution": label_distribution,
    "annotation_csv": csv_path.name,
    "annotation_csv_sha256": file_hash(csv_path),
    "reference_labels_accessed": False,
    "correction_performed": False
}

report_path = (
    AUDIT_DIR / "development_annotation_audit_v1.json"
)

report_content = json.dumps(
    audit_report, indent=2
) + "\n"

report_status = save_immutable(
    report_path, report_content
)

print("\nSTAGE 12 — FULL DEVELOPMENT ANNOTATION AUDIT")
print("Expected records:", 1800)
print("Verified records:", len(audited))
print("Missing checkpoints:", 0)
print("Invalid predictions:", 0)
print("Smoke predictions reused:", 8)

print("\nLLM LABEL DISTRIBUTION")
display(pd.DataFrame([
    {
        "class_id": class_id,
        "class_name": [
            "World", "Sports", "Business", "Sci/Tech"
        ][class_id],
        "predictions": label_distribution[str(class_id)]
    }
    for class_id in range(4)
]))

print("\nSAVED ARTIFACTS")
print("Annotation CSV:", csv_status)
print("CSV path:", csv_path)
print("Audit report:", report_status)
print("Report path:", report_path)
print("Reference labels accessed: NO")
print("New LLM inference performed: NO")
print("Correction performed: NO")

print("\nSTATUS: FULL DEVELOPMENT ANNOTATION AUDIT PASSED")


Audited: 300/1800
Audited: 600/1800
Audited: 900/1800
Audited: 1200/1800
Audited: 1500/1800
Audited: 1800/1800

STAGE 12 — FULL DEVELOPMENT ANNOTATION AUDIT
Expected records: 1800
Verified records: 1800
Missing checkpoints: 0
Invalid predictions: 0
Smoke predictions reused: 8

LLM LABEL DISTRIBUTION


,class_id,class_name,predictions
0,0,World,545
1,1,Sports,466
2,2,Business,546
3,3,Sci/Tech,243



SAVED ARTIFACTS
Annotation CSV: NEW_SAVED
CSV path: /content/drive/MyDrive/S_SLTE_Thesis/outputs/llm_development_v1/development_annotations_v1.csv
Audit report: NEW_SAVED
Report path: /content/drive/MyDrive/S_SLTE_Thesis/outputs/llm_development_v1/development_annotation_audit_v1.json
Reference labels accessed: NO
New LLM inference performed: NO
Correction performed: NO

STATUS: FULL DEVELOPMENT ANNOTATION AUDIT PASSED


In [11]:

import json
import os
import hashlib

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

DEV_DIR = BASE / "outputs/llm_development_v1"
EVIDENCE_DIR = BASE / "outputs/s_slte_evidence_v1"
CAL_DIR = BASE / "outputs/s_slte_calibration_v1"
CAL_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_SEED = 42
CALIBRATION_SIZE = 1080
EVALUATION_SIZE = 720

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

def fingerprint_ids(ids):
    payload = "\n".join(
        str(int(x)) for x in sorted(ids)
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()

# Verify the final LLM annotation audit and CSV.
with (DEV_DIR / "development_annotation_audit_v1.json").open(
    encoding="utf-8"
) as f:
    dev_audit = json.load(f)

assert dev_audit["status"] == (
    "FULL_DEVELOPMENT_ANNOTATION_AUDIT_PASSED"
)
assert dev_audit["verified_records"] == 1800
assert dev_audit["invalid_predictions"] == 0

annotation_path = DEV_DIR / "development_annotations_v1.csv"

assert sha256_file(annotation_path) == (
    dev_audit["annotation_csv_sha256"]
)

annotations = pd.read_csv(annotation_path)

# Verify the previously saved semantic-neighborhood evidence.
with (EVIDENCE_DIR / "evidence_manifest_v1.json").open(
    encoding="utf-8"
) as f:
    evidence_manifest = json.load(f)

evidence_path = (
    EVIDENCE_DIR / "development_neighbor_evidence_v1.csv"
)

assert sha256_file(evidence_path) == (
    evidence_manifest["partitions"]["development"]["sha256"]
)

evidence = pd.read_csv(evidence_path)

assert len(annotations) == len(evidence) == 1800
assert annotations["record_id"].is_unique
assert evidence["record_id"].is_unique
assert annotations["llm_label"].isin(range(4)).all()

# Join predictions with semantic evidence, without reference labels.
prediction_df = annotations.merge(
    evidence,
    on="record_id",
    validate="one_to_one"
).sort_values("record_id").reset_index(drop=True)

assert len(prediction_df) == 1800
assert prediction_df["has_positive_similarity"].all()
assert prediction_df["neighbor_top_label"].isin(range(4)).all()

all_ids = prediction_df["record_id"].to_numpy(dtype=np.int64)

# Freeze a reproducible split without inspecting ground-truth labels.
calibration_ids, evaluation_ids = train_test_split(
    all_ids,
    train_size=CALIBRATION_SIZE,
    test_size=EVALUATION_SIZE,
    random_state=SPLIT_SEED,
    shuffle=True
)

calibration_ids = sorted(map(int, calibration_ids))
evaluation_ids = sorted(map(int, evaluation_ids))

assert len(calibration_ids) == 1080
assert len(evaluation_ids) == 720
assert not set(calibration_ids).intersection(evaluation_ids)
assert set(calibration_ids) | set(evaluation_ids) == set(all_ids)

split_protocol = {
    "protocol_version": "v1",
    "purpose": "EXPLORATORY_DEVELOPMENT_CALIBRATION",
    "seed": SPLIT_SEED,
    "split_method": "RANDOM_RECORD_ID_SPLIT",
    "ground_truth_used_for_split": False,
    "calibration_count": 1080,
    "evaluation_count": 720,
    "calibration_ids": calibration_ids,
    "evaluation_ids": evaluation_ids,
    "calibration_id_sha256": fingerprint_ids(calibration_ids),
    "evaluation_id_sha256": fingerprint_ids(evaluation_ids),
    "annotation_csv_sha256": sha256_file(annotation_path),
    "neighbor_evidence_sha256": sha256_file(evidence_path),
    "evaluation_status": "NOT_EVALUATED",
    "limitations": (
        "Full-development neighborhood diagnostics were explored "
        "before this split. The evaluation subset is not a "
        "fully untouched independent test set."
    )
}

split_path = CAL_DIR / "development_split_v1.json"
split_content = json.dumps(split_protocol, indent=2) + "\n"

if split_path.exists():
    assert split_path.read_text(
        encoding="utf-8"
    ) == split_content
    split_status = "EXISTING_VERIFIED"
else:
    temporary = split_path.with_suffix(".json.tmp")
    temporary.write_text(split_content, encoding="utf-8")
    os.replace(temporary, split_path)
    split_status = "NEW_SAVED"

# Use reference labels only for the calibration subset.
development_reference = pd.read_csv(
    BASE / "data/splits/development.csv",
    usecols=["record_id", "label"]
)

calibration_reference = development_reference.loc[
    development_reference["record_id"].isin(calibration_ids)
].copy()

calibration_df = prediction_df.loc[
    prediction_df["record_id"].isin(calibration_ids)
].merge(
    calibration_reference,
    on="record_id",
    validate="one_to_one"
).sort_values("record_id").reset_index(drop=True)

del development_reference, calibration_reference

assert len(calibration_df) == 1080
assert calibration_df["label"].isin(range(4)).all()

truth = calibration_df["label"].to_numpy(dtype=int)
llm = calibration_df["llm_label"].to_numpy(dtype=int)
neighbor = calibration_df["neighbor_top_label"].to_numpy(dtype=int)

conflict = llm != neighbor
llm_correct = llm == truth
neighbor_correct = neighbor == truth

# Distinguish potentially helpful and harmful label replacements.
beneficial = conflict & ~llm_correct & neighbor_correct
harmful = conflict & llm_correct & ~neighbor_correct
both_wrong = conflict & ~llm_correct & ~neighbor_correct

assert (
    int(beneficial.sum())
    + int(harmful.sum())
    + int(both_wrong.sum())
) == int(conflict.sum())

baseline_accuracy = accuracy_score(truth, llm)
baseline_macro_f1 = f1_score(
    truth, llm, labels=[0, 1, 2, 3],
    average="macro", zero_division=0
)

neighbor_accuracy = accuracy_score(truth, neighbor)

diagnostics = {
    "protocol_version": "v1",
    "partition": "calibration_only",
    "records": 1080,
    "original_llm_accuracy": round(
        float(baseline_accuracy), 6
    ),
    "original_llm_macro_f1": round(
        float(baseline_macro_f1), 6
    ),
    "neighbor_accuracy": round(
        float(neighbor_accuracy), 6
    ),
    "llm_neighbor_conflicts": int(conflict.sum()),
    "potentially_beneficial_replacements": int(beneficial.sum()),
    "potentially_harmful_replacements": int(harmful.sum()),
    "both_labels_wrong": int(both_wrong.sum()),
    "threshold_selected": False,
    "correction_performed": False,
    "evaluation_subset_scored": False,
    "holdout_reference_labels_accessed": False,
    "split_manifest_sha256": sha256_file(split_path)
}

report_path = CAL_DIR / "calibration_baseline_v1.json"
report_content = json.dumps(diagnostics, indent=2) + "\n"

if report_path.exists():
    assert report_path.read_text(
        encoding="utf-8"
    ) == report_content
    report_status = "EXISTING_VERIFIED"
else:
    temporary = report_path.with_suffix(".json.tmp")
    temporary.write_text(report_content, encoding="utf-8")
    os.replace(temporary, report_path)
    report_status = "NEW_SAVED"

print("STAGE 12 — DEVELOPMENT CALIBRATION BASELINE")
print("Split:", split_status)
print("Calibration records:", len(calibration_df))
print("Reserved evaluation records:", len(evaluation_ids))

print(f"\nOriginal LLM accuracy: {baseline_accuracy:.4%}")
print(f"Original LLM Macro-F1: {baseline_macro_f1:.4f}")
print(f"Neighborhood accuracy: {neighbor_accuracy:.4%}")

print("\nCALIBRATION CONFLICT ANALYSIS")
print("LLM–Neighbor conflicts:", int(conflict.sum()))
print("Potentially beneficial:", int(beneficial.sum()))
print("Potentially harmful:", int(harmful.sum()))
print("Both labels incorrect:", int(both_wrong.sum()))

print("\nREPORT")
print("Baseline report:", report_status)
print("Threshold selected: NO")
print("Correction performed: NO")
print("Evaluation subset accuracy calculated: NO")
print("Frozen holdout reference labels accessed: NO")
print("\nSTATUS: CALIBRATION BASELINE VERIFIED")


STAGE 12 — DEVELOPMENT CALIBRATION BASELINE
Split: NEW_SAVED
Calibration records: 1080
Reserved evaluation records: 720

Original LLM accuracy: 80.0926%
Original LLM Macro-F1: 0.7908
Neighborhood accuracy: 87.8704%

CALIBRATION CONFLICT ANALYSIS
LLM–Neighbor conflicts: 212
Potentially beneficial: 136
Potentially harmful: 52
Both labels incorrect: 24

REPORT
Baseline report: NEW_SAVED
Threshold selected: NO
Correction performed: NO
Evaluation subset accuracy calculated: NO
Frozen holdout reference labels accessed: NO

STATUS: CALIBRATION BASELINE VERIFIED


In [12]:

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

CAL_DIR = BASE / "outputs/s_slte_calibration_v1"

with (CAL_DIR / "development_split_v1.json").open(
    encoding="utf-8"
) as f:
    split_manifest = json.load(f)

with (CAL_DIR / "calibration_baseline_v1.json").open(
    encoding="utf-8"
) as f:
    baseline_report = json.load(f)

assert len(calibration_df) == 1080
assert baseline_report["records"] == 1080
assert baseline_report["llm_neighbor_conflicts"] == 212
assert baseline_report["potentially_beneficial_replacements"] == 136
assert baseline_report["potentially_harmful_replacements"] == 52
assert baseline_report["both_labels_wrong"] == 24

assert set(calibration_df["record_id"].astype(int)) == set(
    split_manifest["calibration_ids"]
)

# Use calibration labels only; evaluation labels remain untouched.
cal = calibration_df.sort_values(
    "record_id"
).reset_index(drop=True)

truth = cal["label"].to_numpy(dtype=int)
llm = cal["llm_label"].to_numpy(dtype=int)
neighbor = cal["neighbor_top_label"].to_numpy(dtype=int)

support = cal[
    [f"support_class_{i}" for i in range(4)]
].to_numpy(dtype=float)

assert support.shape == (1080, 4)
assert np.isfinite(support).all()
assert np.allclose(support.sum(axis=1), 1.0, atol=1e-6)
assert np.array_equal(support.argmax(axis=1), neighbor)

top_support = support.max(axis=1)
llm_support = support[np.arange(len(llm)), llm]
support_gap = top_support - llm_support

conflict = neighbor != llm
llm_correct = llm == truth
neighbor_correct = neighbor == truth

assert int(llm_correct.sum()) == 865
assert int(neighbor_correct.sum()) == 949
assert int(conflict.sum()) == 212

def wilson_upper(errors, count, z=1.959963984540054):
    if count == 0:
        return None

    p = errors / count
    denominator = 1 + z**2 / count
    center = (p + z**2 / (2 * count)) / denominator

    radius = (
        z * np.sqrt(
            p * (1 - p) / count + z**2 / (4 * count**2)
        ) / denominator
    )

    return round(float(min(1, center + radius)), 6)

SUPPORT_THRESHOLDS = [0.70, 0.80, 0.90]
GAP_THRESHOLDS = [0.50, 0.65, 0.80]

results = []
class_results = []

class_names = ["World", "Sports", "Business", "Sci/Tech"]

for min_support in SUPPORT_THRESHOLDS:
    for min_gap in GAP_THRESHOLDS:

        selected = (
            conflict
            & (top_support >= min_support)
            & (support_gap >= min_gap)
        )

        beneficial = selected & ~llm_correct & neighbor_correct
        harmful = selected & llm_correct & ~neighbor_correct
        both_wrong = selected & ~llm_correct & ~neighbor_correct

        count = int(selected.sum())
        benefit_count = int(beneficial.sum())
        harm_count = int(harmful.sum())
        both_wrong_count = int(both_wrong.sum())

        assert (
            benefit_count + harm_count + both_wrong_count
        ) == count

        corrected = llm.copy()
        corrected[selected] = neighbor[selected]

        corrected_accuracy = float(
            (corrected == truth).mean()
        )

        expected_correct = (
            int(llm_correct.sum()) + benefit_count - harm_count
        )

        assert int((corrected == truth).sum()) == expected_correct

        results.append({
            "min_support": min_support,
            "min_gap": min_gap,
            "candidates": count,
            "beneficial": benefit_count,
            "harmful": harm_count,
            "both_wrong": both_wrong_count,
            "beneficial_fraction": (
                round(benefit_count / count, 6)
                if count else None
            ),
            "harmful_fraction": (
                round(harm_count / count, 6)
                if count else None
            ),
            "harmful_wilson_95_upper": wilson_upper(
                harm_count, count
            ),
            "net_correct_gain": benefit_count - harm_count,
            "net_accuracy_gain_pp": round(
                100 * (benefit_count - harm_count) / 1080,
                4
            ),
            "resulting_accuracy": round(
                corrected_accuracy, 6
            ),
            "abstained_conflicts": int(conflict.sum()) - count
        })

        for class_id, class_name in enumerate(class_names):
            class_selected = selected & (neighbor == class_id)

            class_results.append({
                "min_support": min_support,
                "min_gap": min_gap,
                "proposed_class": class_name,
                "candidates": int(class_selected.sum()),
                "beneficial": int(
                    (class_selected & beneficial).sum()
                ),
                "harmful": int(
                    (class_selected & harmful).sum()
                ),
                "both_wrong": int(
                    (class_selected & both_wrong).sum()
                )
            })

diagnostics = pd.DataFrame(results)
class_diagnostics = pd.DataFrame(class_results)

assert len(diagnostics) == 9
assert len(class_diagnostics) == 36
assert (diagnostics["candidates"] <= 212).all()
assert (
    diagnostics["net_correct_gain"]
    == diagnostics["beneficial"] - diagnostics["harmful"]
).all()

report = {
    "protocol_version": "v1",
    "analysis": "EXPLORATORY_CORRECTION_CANDIDATE_DIAGNOSTICS",
    "partition": "calibration_only",
    "records": 1080,
    "original_llm_correct": 865,
    "original_llm_accuracy": round(865 / 1080, 6),
    "neighbor_correct": 949,
    "total_conflicts": 212,
    "rule": (
        "Replace LLM label with neighbor top class only "
        "when labels conflict and both thresholds pass; "
        "otherwise retain original LLM label."
    ),
    "support_thresholds": SUPPORT_THRESHOLDS,
    "gap_thresholds": GAP_THRESHOLDS,
    "candidate_results": results,
    "class_conditional_results": class_results,
    "wilson_interval_convention": (
        "Upper endpoint of two-sided 95% Wilson interval"
    ),
    "threshold_selected": False,
    "training_correction_applied": False,
    "evaluation_subset_scored": False,
    "holdout_reference_labels_accessed": False,
    "limitations": (
        "Exploratory calibration-set results. Thresholds are "
        "not selected and reported intervals are unadjusted "
        "for multiple candidate-rule comparisons."
    ),
    "split_manifest_sha256": sha256_file(
        CAL_DIR / "development_split_v1.json"
    ),
    "baseline_report_sha256": sha256_file(
        CAL_DIR / "calibration_baseline_v1.json"
    )
}

report_path = (
    CAL_DIR / "correction_candidate_diagnostics_v1.json"
)

payload = json.dumps(report, indent=2) + "\n"

if report_path.exists():
    assert report_path.read_text(
        encoding="utf-8"
    ) == payload
    status = "EXISTING_VERIFIED"
else:
    temporary = report_path.with_suffix(".json.tmp")
    temporary.write_text(payload, encoding="utf-8")
    os.replace(temporary, report_path)
    status = "NEW_SAVED"

print("STAGE 12 — CORRECTION CANDIDATE DIAGNOSTICS")

print("\nCALIBRATION RULE COMPARISON")
display(diagnostics)

print("\nILLUSTRATIVE CLASS ANALYSIS — SUPPORT 0.80, GAP 0.65")
display(
    class_diagnostics.loc[
        (class_diagnostics["min_support"] == 0.80)
        & (class_diagnostics["min_gap"] == 0.65)
    ].reset_index(drop=True)
)

print("\nCalibration records:", len(cal))
print("Original correct predictions:", int(llm_correct.sum()))
print("Total conflicts:", int(conflict.sum()))
print("Report:", status)
print("Saved path:", report_path)
print("Threshold selected: NO")
print("Training corrections applied: NO")
print("Evaluation subset scored: NO")
print("Holdout reference labels accessed: NO")
print("\nSTATUS: CORRECTION CANDIDATE DIAGNOSTICS PASSED")


STAGE 12 — CORRECTION CANDIDATE DIAGNOSTICS

CALIBRATION RULE COMPARISON


,min_support,min_gap,candidates,beneficial,harmful,both_wrong,beneficial_fraction,harmful_fraction,harmful_wilson_95_upper,net_correct_gain,net_accuracy_gain_pp,resulting_accuracy,abstained_conflicts
0,0.7,0.50,78,67,8,3,0.858974,0.102564,0.189544,59,5.4630,0.855556,134
1,0.7,0.65,57,51,4,2,0.894737,0.070175,0.167002,47,4.3519,0.844444,155
2,0.7,0.80,30,28,2,0,0.933333,0.066667,0.213235,26,2.4074,0.825000,182
3,0.8,0.50,52,49,3,0,0.942308,0.057692,0.156425,46,4.2593,0.843519,160
4,0.8,0.65,50,47,3,0,0.940000,0.060000,0.162171,44,4.0741,0.841667,162
5,0.8,0.80,30,28,2,0,0.933333,0.066667,0.213235,26,2.4074,0.825000,182
6,0.9,0.50,21,21,0,0,1.000000,0.000000,0.154639,21,1.9444,0.820370,191
7,0.9,0.65,21,21,0,0,1.000000,0.000000,0.154639,21,1.9444,0.820370,191
8,0.9,0.80,21,21,0,0,1.000000,0.000000,0.154639,21,1.9444,0.820370,191



ILLUSTRATIVE CLASS ANALYSIS — SUPPORT 0.80, GAP 0.65


,min_support,min_gap,proposed_class,candidates,beneficial,harmful,both_wrong
0,0.8,0.65,World,0,0,0,0
1,0.8,0.65,Sports,3,3,0,0
2,0.8,0.65,Business,9,7,2,0
3,0.8,0.65,Sci/Tech,38,37,1,0



Calibration records: 1080
Original correct predictions: 865
Total conflicts: 212
Report: NEW_SAVED
Saved path: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_calibration_v1/correction_candidate_diagnostics_v1.json
Threshold selected: NO
Training corrections applied: NO
Evaluation subset scored: NO
Holdout reference labels accessed: NO

STATUS: CORRECTION CANDIDATE DIAGNOSTICS PASSED


In [13]:

from pathlib import Path
import json
import os
import hashlib

import pandas as pd

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
CAL_DIR = BASE / "outputs/s_slte_calibration_v1"

MIN_SUPPORT = 0.80
MIN_GAP = 0.50

def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

# Verify the previously saved calibration artifacts.
split_path = CAL_DIR / "development_split_v1.json"
baseline_path = CAL_DIR / "calibration_baseline_v1.json"
diagnostics_path = (
    CAL_DIR / "correction_candidate_diagnostics_v1.json"
)

with split_path.open(encoding="utf-8") as f:
    split_manifest = json.load(f)

with baseline_path.open(encoding="utf-8") as f:
    baseline = json.load(f)

with diagnostics_path.open(encoding="utf-8") as f:
    diagnostics = json.load(f)

assert split_manifest["calibration_count"] == 1080
assert split_manifest["evaluation_count"] == 720
assert split_manifest["evaluation_status"] == "NOT_EVALUATED"

assert baseline["records"] == 1080
assert baseline["evaluation_subset_scored"] is False
assert diagnostics["threshold_selected"] is False
assert diagnostics["evaluation_subset_scored"] is False
assert diagnostics["split_manifest_sha256"] == (
    sha256_file(split_path)
)
assert diagnostics["baseline_report_sha256"] == (
    sha256_file(baseline_path)
)

# Select exactly one previously evaluated candidate rule.
matches = [
    row
    for row in diagnostics["candidate_results"]
    if row["min_support"] == MIN_SUPPORT
    and row["min_gap"] == MIN_GAP
]

assert len(matches) == 1
selected = matches[0]

assert selected["candidates"] == 52
assert selected["beneficial"] == 49
assert selected["harmful"] == 3
assert selected["both_wrong"] == 0
assert selected["net_correct_gain"] == 46

# Freeze the selection without inspecting evaluation labels.
frozen_rule = {
    "protocol_version": "v1",
    "method": "S_SLTE_SEMANTIC_ONLY_CONSERVATIVE_RULE",
    "selection_status": "FROZEN_AFTER_EXPLORATORY_CALIBRATION",
    "selection_basis": "POST_HOC_CALIBRATION_RESULTS",
    "min_neighbor_support": MIN_SUPPORT,
    "min_semantic_support_gap": MIN_GAP,
    "k": 20,
    "neighbor_pool": "CLEAN_ANCHOR_ONLY",
    "neighbor_weighting": "MAX_COSINE_ZERO",
    "decision": (
        "Replace the original LLM label with the neighborhood "
        "top class only if labels disagree, neighborhood top "
        "support >= 0.80, and top support minus LLM-label "
        "support >= 0.50. Otherwise retain the LLM label."
    ),
    "abstention_action": "RETAIN_ORIGINAL_LLM_LABEL",
    "class_specific_thresholds": False,
    "calibration_records": 1080,
    "reserved_development_evaluation_records": 720,
    "observed_calibration_candidates": 52,
    "observed_calibration_beneficial": 49,
    "observed_calibration_harmful": 3,
    "observed_calibration_net_gain": 46,
    "evaluation_subset_scored": False,
    "training_labels_modified": False,
    "holdout_reference_labels_accessed": False,
    "independence_limitation": (
        "The full development set was used in earlier "
        "exploratory neighborhood diagnostics. The reserved "
        "evaluation subset is therefore not completely untouched."
    ),
    "source_sha256": {
        "development_split": sha256_file(split_path),
        "calibration_baseline": sha256_file(baseline_path),
        "candidate_diagnostics": sha256_file(diagnostics_path)
    }
}

rule_path = CAL_DIR / "frozen_correction_rule_v1.json"
content = json.dumps(frozen_rule, indent=2) + "\n"

if rule_path.exists():
    assert rule_path.read_text(encoding="utf-8") == content
    status = "EXISTING_VERIFIED"
else:
    temporary = rule_path.with_suffix(".json.tmp")
    temporary.write_text(content, encoding="utf-8")
    os.replace(temporary, rule_path)
    status = "NEW_SAVED"

print("STAGE 12 — CONSERVATIVE CORRECTION RULE FROZEN")
print("Method:", frozen_rule["method"])
print("Minimum support:", MIN_SUPPORT)
print("Minimum support gap:", MIN_GAP)
print("Neighbor count:", frozen_rule["k"])
print("Calibration candidates:", selected["candidates"])
print("Beneficial:", selected["beneficial"])
print("Harmful:", selected["harmful"])
print("Net correct gain:", selected["net_correct_gain"])
print("Rule:", status)
print("Saved path:", rule_path)
print("Evaluation subset scored: NO")
print("Training labels modified: NO")
print("Holdout reference labels accessed: NO")
print("\nSTATUS: S-SLTE RULE V1 FROZEN")


STAGE 12 — CONSERVATIVE CORRECTION RULE FROZEN
Method: S_SLTE_SEMANTIC_ONLY_CONSERVATIVE_RULE
Minimum support: 0.8
Minimum support gap: 0.5
Neighbor count: 20
Calibration candidates: 52
Beneficial: 49
Harmful: 3
Net correct gain: 46
Rule: NEW_SAVED
Saved path: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_calibration_v1/frozen_correction_rule_v1.json
Evaluation subset scored: NO
Training labels modified: NO
Holdout reference labels accessed: NO

STATUS: S-SLTE RULE V1 FROZEN


In [14]:

from pathlib import Path
import json
import os
import hashlib

import numpy as np
import pandas as pd

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
CAL_DIR = BASE / "outputs/s_slte_calibration_v1"
DEV_DIR = BASE / "outputs/llm_development_v1"
EVIDENCE_DIR = BASE / "outputs/s_slte_evidence_v1"

def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

def save_immutable(path, content):
    payload = content.encode("utf-8")

    if path.exists():
        assert path.read_bytes() == payload, (
            f"Existing artifact differs: {path}"
        )
        return "EXISTING_VERIFIED"

    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_bytes(payload)
    os.replace(temporary, path)

    assert path.read_bytes() == payload
    return "NEW_SAVED"

split_path = CAL_DIR / "development_split_v1.json"
rule_path = CAL_DIR / "frozen_correction_rule_v1.json"

with split_path.open(encoding="utf-8") as f:
    split_manifest = json.load(f)

with rule_path.open(encoding="utf-8") as f:
    frozen_rule = json.load(f)

with (
    DEV_DIR / "development_annotation_audit_v1.json"
).open(encoding="utf-8") as f:
    dev_audit = json.load(f)

with (
    EVIDENCE_DIR / "evidence_manifest_v1.json"
).open(encoding="utf-8") as f:
    evidence_manifest = json.load(f)

assert frozen_rule["method"] == (
    "S_SLTE_SEMANTIC_ONLY_CONSERVATIVE_RULE"
)
assert frozen_rule["selection_status"] == (
    "FROZEN_AFTER_EXPLORATORY_CALIBRATION"
)
assert frozen_rule["min_neighbor_support"] == 0.80
assert frozen_rule["min_semantic_support_gap"] == 0.50
assert frozen_rule["k"] == 20
assert frozen_rule["evaluation_subset_scored"] is False

assert frozen_rule["source_sha256"]["development_split"] == (
    sha256_file(split_path)
)

assert split_manifest["calibration_count"] == 1080
assert split_manifest["evaluation_count"] == 720

evaluation_ids = set(
    map(int, split_manifest["evaluation_ids"])
)
calibration_ids = set(
    map(int, split_manifest["calibration_ids"])
)

assert len(evaluation_ids) == 720
assert len(calibration_ids) == 1080
assert evaluation_ids.isdisjoint(calibration_ids)

annotation_path = DEV_DIR / "development_annotations_v1.csv"
evidence_path = (
    EVIDENCE_DIR / "development_neighbor_evidence_v1.csv"
)

assert sha256_file(annotation_path) == (
    dev_audit["annotation_csv_sha256"]
)
assert sha256_file(evidence_path) == (
    evidence_manifest["partitions"]["development"]["sha256"]
)

annotations = pd.read_csv(annotation_path)
evidence = pd.read_csv(evidence_path)

assert len(annotations) == len(evidence) == 1800
assert annotations["record_id"].is_unique
assert evidence["record_id"].is_unique

inputs = annotations.merge(
    evidence,
    on="record_id",
    how="inner",
    validate="one_to_one"
)

evaluation = (
    inputs.loc[inputs["record_id"].isin(evaluation_ids)]
    .sort_values("record_id")
    .reset_index(drop=True)
)

assert len(evaluation) == 720
assert set(evaluation["record_id"].astype(int)) == evaluation_ids
assert "label" not in evaluation.columns

support = evaluation[
    [f"support_class_{i}" for i in range(4)]
].to_numpy(dtype=np.float64)

llm_labels = evaluation["llm_label"].to_numpy(dtype=int)
neighbor_labels = evaluation[
    "neighbor_top_label"
].to_numpy(dtype=int)

assert np.isin(llm_labels, range(4)).all()
assert np.isin(neighbor_labels, range(4)).all()
assert np.isfinite(support).all()
assert np.allclose(support.sum(axis=1), 1.0, atol=1e-6)
assert np.array_equal(support.argmax(axis=1), neighbor_labels)

top_support = support.max(axis=1)
llm_support = support[
    np.arange(len(evaluation)), llm_labels
]

support_gap = np.maximum(
    top_support - llm_support, 0.0
)

assert np.allclose(
    top_support,
    evaluation["neighbor_top_support"].to_numpy(),
    atol=1e-6
)

conflict = llm_labels != neighbor_labels

should_correct = (
    conflict
    & (top_support >= frozen_rule["min_neighbor_support"])
    & (support_gap >= frozen_rule["min_semantic_support_gap"])
)

corrected_labels = np.where(
    should_correct,
    neighbor_labels,
    llm_labels
)

assert np.all(corrected_labels[~should_correct] == (
    llm_labels[~should_correct]
))
assert np.all(corrected_labels[should_correct] == (
    neighbor_labels[should_correct]
))

decisions = pd.DataFrame({
    "record_id": evaluation["record_id"].to_numpy(dtype=np.int64),
    "original_llm_label": llm_labels,
    "neighbor_proposed_label": neighbor_labels,
    "neighbor_top_support": top_support,
    "llm_label_support": llm_support,
    "semantic_support_gap": support_gap,
    "llm_neighbor_conflict": conflict,
    "correction_applied": should_correct,
    "corrected_label": corrected_labels
})

assert decisions["record_id"].is_unique
assert len(decisions) == 720
assert "label" not in decisions.columns

decisions_path = CAL_DIR / "evaluation_decisions_v1.csv"

content = decisions.to_csv(
    index=False,
    float_format="%.8f",
    lineterminator="\n"
)

decision_status = save_immutable(decisions_path, content)

decision_manifest = {
    "protocol_version": "v1",
    "stage": "FROZEN_RULE_EVALUATION_DECISIONS",
    "method": frozen_rule["method"],
    "evaluation_records": 720,
    "neighbor_conflicts": int(conflict.sum()),
    "corrections_applied": int(should_correct.sum()),
    "abstained_conflicts": int(
        (conflict & ~should_correct).sum()
    ),
    "unchanged_records": int((~should_correct).sum()),
    "min_support": 0.80,
    "min_gap": 0.50,
    "rule_sha256": sha256_file(rule_path),
    "split_sha256": sha256_file(split_path),
    "annotation_sha256": sha256_file(annotation_path),
    "evidence_sha256": sha256_file(evidence_path),
    "decisions_csv_sha256": sha256_file(decisions_path),
    "reference_labels_used_for_decisions": False,
    "evaluation_accuracy_calculated": False,
    "training_labels_modified": False,
    "holdout_reference_labels_accessed": False
}

manifest_path = CAL_DIR / "evaluation_decision_manifest_v1.json"

manifest_content = json.dumps(
    decision_manifest, indent=2
) + "\n"

manifest_status = save_immutable(
    manifest_path, manifest_content
)

print("STAGE 12 — FROZEN RULE EVALUATION DECISIONS")
print("Evaluation records:", len(decisions))
print("LLM–Neighbor conflicts:", int(conflict.sum()))
print("Corrections proposed:", int(should_correct.sum()))
print(
    "Conflicts abstained:",
    int((conflict & ~should_correct).sum())
)
print("Original labels retained:", int((~should_correct).sum()))

print("\nARTIFACTS")
print("Decisions:", decision_status)
print("Decision manifest:", manifest_status)
print("Frozen rule fingerprint: VERIFIED")
print("Source files: VERIFIED")
print("Reference labels used for decisions: NO")
print("Evaluation accuracy calculated: NO")
print("Training labels modified: NO")
print("Holdout reference labels accessed: NO")
print("\nSTATUS: FROZEN EVALUATION DECISIONS SAVED")


STAGE 12 — FROZEN RULE EVALUATION DECISIONS
Evaluation records: 720
LLM–Neighbor conflicts: 137
Corrections proposed: 36
Conflicts abstained: 101
Original labels retained: 684

ARTIFACTS
Decisions: NEW_SAVED
Decision manifest: NEW_SAVED
Frozen rule fingerprint: VERIFIED
Source files: VERIFIED
Reference labels used for decisions: NO
Evaluation accuracy calculated: NO
Training labels modified: NO
Holdout reference labels accessed: NO

STATUS: FROZEN EVALUATION DECISIONS SAVED


In [15]:

import os
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score
from scipy.stats import binomtest

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
CAL_DIR = BASE / "outputs/s_slte_calibration_v1"
DEV_DIR = BASE / "outputs/llm_development_v1"
EVIDENCE_DIR = BASE / "outputs/s_slte_evidence_v1"

def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

def save_immutable(path, content):
    payload = content.encode("utf-8")

    if path.exists():
        assert path.read_bytes() == payload, (
            f"Existing artifact differs: {path}"
        )
        return "EXISTING_VERIFIED"

    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_bytes(payload)
    os.replace(temporary, path)

    return "NEW_SAVED"

split_path = CAL_DIR / "development_split_v1.json"
rule_path = CAL_DIR / "frozen_correction_rule_v1.json"
decisions_path = CAL_DIR / "evaluation_decisions_v1.csv"
decision_manifest_path = (
    CAL_DIR / "evaluation_decision_manifest_v1.json"
)

annotation_path = DEV_DIR / "development_annotations_v1.csv"
evidence_path = (
    EVIDENCE_DIR / "development_neighbor_evidence_v1.csv"
)
reference_path = BASE / "data/splits/development.csv"

with split_path.open(encoding="utf-8") as f:
    split = json.load(f)

with rule_path.open(encoding="utf-8") as f:
    rule = json.load(f)

with decision_manifest_path.open(encoding="utf-8") as f:
    decision_manifest = json.load(f)

assert rule["min_neighbor_support"] == 0.80
assert rule["min_semantic_support_gap"] == 0.50
assert rule["k"] == 20
assert rule["evaluation_subset_scored"] is False

assert decision_manifest["rule_sha256"] == sha256_file(rule_path)
assert decision_manifest["split_sha256"] == sha256_file(split_path)
assert decision_manifest["decisions_csv_sha256"] == (
    sha256_file(decisions_path)
)
assert decision_manifest["annotation_sha256"] == (
    sha256_file(annotation_path)
)
assert decision_manifest["evidence_sha256"] == (
    sha256_file(evidence_path)
)

assert decision_manifest["evaluation_records"] == 720
assert decision_manifest["corrections_applied"] == 36
assert decision_manifest["neighbor_conflicts"] == 137
assert decision_manifest["evaluation_accuracy_calculated"] is False

evaluation_ids = set(map(int, split["evaluation_ids"]))
calibration_ids = set(map(int, split["calibration_ids"]))

assert len(evaluation_ids) == 720
assert len(calibration_ids) == 1080
assert evaluation_ids.isdisjoint(calibration_ids)

# Audit frozen decisions before accessing reference labels.
decisions = pd.read_csv(decisions_path)
annotations = pd.read_csv(annotation_path)
evidence = pd.read_csv(evidence_path)

assert len(decisions) == 720
assert decisions["record_id"].is_unique
assert set(decisions["record_id"].astype(int)) == evaluation_ids
assert pd.api.types.is_bool_dtype(
    decisions["correction_applied"]
)
assert pd.api.types.is_bool_dtype(
    decisions["llm_neighbor_conflict"]
)

source = annotations.merge(
    evidence,
    on="record_id",
    validate="one_to_one"
)

source = (
    source.loc[source["record_id"].isin(evaluation_ids)]
    .sort_values("record_id")
    .reset_index(drop=True)
)

decisions = decisions.sort_values(
    "record_id"
).reset_index(drop=True)

assert np.array_equal(
    decisions["record_id"].to_numpy(),
    source["record_id"].to_numpy()
)

llm = source["llm_label"].to_numpy(dtype=int)
neighbor = source["neighbor_top_label"].to_numpy(dtype=int)

support = source[
    [f"support_class_{i}" for i in range(4)]
].to_numpy(dtype=float)

assert support.shape == (720, 4)
assert np.isfinite(support).all()
assert np.allclose(support.sum(axis=1), 1, atol=1e-6)
assert np.array_equal(support.argmax(axis=1), neighbor)

top_support = support.max(axis=1)
llm_support = support[np.arange(720), llm]
gap = np.maximum(top_support - llm_support, 0.0)

conflict = llm != neighbor

expected_selected = (
    conflict
    & (top_support >= rule["min_neighbor_support"])
    & (gap >= rule["min_semantic_support_gap"])
)

expected_corrected = np.where(
    expected_selected, neighbor, llm
)

selected = decisions[
    "correction_applied"
].to_numpy(dtype=bool)

corrected = decisions[
    "corrected_label"
].to_numpy(dtype=int)

assert np.array_equal(
    decisions["original_llm_label"].to_numpy(dtype=int),
    llm
)
assert np.array_equal(
    decisions["neighbor_proposed_label"].to_numpy(dtype=int),
    neighbor
)
assert np.array_equal(
    decisions["llm_neighbor_conflict"].to_numpy(dtype=bool),
    conflict
)
assert np.array_equal(selected, expected_selected)
assert np.array_equal(corrected, expected_corrected)
assert int(selected.sum()) == 36

# Access development reference labels only after verification.
reference = pd.read_csv(
    reference_path,
    usecols=["record_id", "label"]
)

assert len(reference) == 1800
assert reference["record_id"].is_unique
assert reference["label"].notna().all()
assert reference["label"].isin([0, 1, 2, 3]).all()

evaluated = decisions.merge(
    reference,
    on="record_id",
    how="left",
    validate="one_to_one"
)

assert len(evaluated) == 720
assert evaluated["label"].notna().all()

truth = evaluated["label"].to_numpy(dtype=int)

original_correct = llm == truth
corrected_correct = corrected == truth
neighbor_correct = neighbor == truth

beneficial = selected & ~original_correct & neighbor_correct
harmful = selected & original_correct & ~neighbor_correct
both_wrong = selected & ~original_correct & ~neighbor_correct

beneficial_count = int(beneficial.sum())
harmful_count = int(harmful.sum())
both_wrong_count = int(both_wrong.sum())

assert (
    beneficial_count + harmful_count + both_wrong_count
) == 36

original_accuracy = accuracy_score(truth, llm)
corrected_accuracy = accuracy_score(truth, corrected)

original_f1 = f1_score(
    truth, llm,
    labels=[0, 1, 2, 3],
    average="macro",
    zero_division=0
)

corrected_f1 = f1_score(
    truth, corrected,
    labels=[0, 1, 2, 3],
    average="macro",
    zero_division=0
)

net_gain = beneficial_count - harmful_count
accuracy_gain_pp = 100 * (
    corrected_accuracy - original_accuracy
)

assert int(corrected_correct.sum()) == (
    int(original_correct.sum()) + net_gain
)

# Exact paired McNemar test for changes in correctness.
discordant = beneficial_count + harmful_count

mcnemar_p = (
    float(
        binomtest(
            beneficial_count,
            discordant,
            p=0.5,
            alternative="two-sided"
        ).pvalue
    )
    if discordant else 1.0
)

# Paired bootstrap confidence interval for accuracy difference.
delta = (
    corrected_correct.astype(int)
    - original_correct.astype(int)
)

rng = np.random.default_rng(20261102)

bootstrap_indices = rng.integers(
    0, 720, size=(10000, 720)
)

bootstrap_gains = (
    delta[bootstrap_indices].mean(axis=1) * 100
)

ci_low, ci_high = np.percentile(
    bootstrap_gains, [2.5, 97.5]
)

class_names = ["World", "Sports", "Business", "Sci/Tech"]

class_results = []

for class_id, class_name in enumerate(class_names):
    mask = selected & (neighbor == class_id)

    class_results.append({
        "proposed_class": class_name,
        "corrections": int(mask.sum()),
        "beneficial": int((mask & beneficial).sum()),
        "harmful": int((mask & harmful).sum()),
        "both_wrong": int((mask & both_wrong).sum())
    })

report = {
    "protocol_version": "v1",
    "stage": "FROZEN_RULE_DEVELOPMENT_EVALUATION",
    "method": rule["method"],
    "evaluation_records": 720,
    "llm_neighbor_conflicts": int(conflict.sum()),
    "corrections_applied": int(selected.sum()),
    "conflicts_abstained": int((conflict & ~selected).sum()),
    "original_labels_retained": int((~selected).sum()),
    "original_correct": int(original_correct.sum()),
    "corrected_correct": int(corrected_correct.sum()),
    "original_accuracy": float(original_accuracy),
    "corrected_accuracy": float(corrected_accuracy),
    "original_macro_f1": float(original_f1),
    "corrected_macro_f1": float(corrected_f1),
    "accuracy_gain_percentage_points": float(accuracy_gain_pp),
    "macro_f1_gain": float(corrected_f1 - original_f1),
    "beneficial_corrections": beneficial_count,
    "harmful_corrections": harmful_count,
    "both_labels_wrong": both_wrong_count,
    "net_correct_gain": net_gain,
    "beneficial_fraction_among_corrections": (
        beneficial_count / 36
    ),
    "harmful_fraction_among_corrections": (
        harmful_count / 36
    ),
    "mcnemar_exact_two_sided_p": mcnemar_p,
    "paired_bootstrap_accuracy_gain_95_ci_pp": [
        float(ci_low), float(ci_high)
    ],
    "bootstrap_resamples": 10000,
    "bootstrap_seed": 20261102,
    "class_conditional_results": class_results,
    "rule_sha256": sha256_file(rule_path),
    "split_sha256": sha256_file(split_path),
    "decisions_sha256": sha256_file(decisions_path),
    "reference_sha256": sha256_file(reference_path),
    "development_reference_labels_accessed": True,
    "holdout_reference_labels_accessed": False,
    "training_labels_modified": False,
    "evaluation_rule_modified": False,
    "independence_limitation": (
        "The full development set was examined during "
        "earlier exploratory neighborhood diagnostics. "
        "This is not a fully untouched independent evaluation."
    ),
    "interpretation_limitations": (
        "Results measure development label correction, not "
        "downstream classifier performance or holdout performance."
    )
}

report_path = (
    CAL_DIR / "development_evaluation_report_v1.json"
)

content = json.dumps(report, indent=2) + "\n"
status = save_immutable(report_path, content)

print("STAGE 12 — FROZEN S-SLTE EVALUATION")

print("\nBEFORE VS AFTER")
print(f"Original LLM accuracy: {original_accuracy:.4%}")
print(f"Corrected accuracy: {corrected_accuracy:.4%}")
print(f"Accuracy gain: {accuracy_gain_pp:+.4f} pp")
print(f"Original Macro-F1: {original_f1:.4f}")
print(f"Corrected Macro-F1: {corrected_f1:.4f}")
print(f"Macro-F1 gain: {corrected_f1 - original_f1:+.4f}")

print("\nCORRECTION SAFETY")
print("Evaluation records:", 720)
print("Corrections applied:", int(selected.sum()))
print("Beneficial:", beneficial_count)
print("Harmful:", harmful_count)
print("Both labels wrong:", both_wrong_count)
print("Net correct gain:", net_gain)
print(
    f"Beneficial fraction: "
    f"{beneficial_count / 36:.2%}"
)
print(
    f"Harmful fraction: "
    f"{harmful_count / 36:.2%}"
)
print("Conflicts abstained:", int((conflict & ~selected).sum()))

print("\nSTATISTICAL DIAGNOSTICS")
print(f"Exact McNemar p-value: {mcnemar_p:.6g}")
print(
    "Paired bootstrap 95% CI for accuracy gain (pp):",
    f"[{ci_low:+.4f}, {ci_high:+.4f}]"
)

print("\nPER-CLASS CORRECTIONS")
display(pd.DataFrame(class_results))

print("\nRESEARCH INTEGRITY")
print("Frozen rule unchanged: YES")
print("Evaluation decisions verified: YES")
print("Development reference labels used: YES")
print("Holdout reference labels accessed: NO")
print("Training labels modified: NO")
print("New LLM inference performed: NO")
print("Report:", status)
print("Saved path:", report_path)
print("\nSTATUS: FROZEN DEVELOPMENT EVALUATION COMPLETED")


STAGE 12 — FROZEN S-SLTE EVALUATION

BEFORE VS AFTER
Original LLM accuracy: 80.1389%
Corrected accuracy: 84.1667%
Accuracy gain: +4.0278 pp
Original Macro-F1: 0.7947
Corrected Macro-F1: 0.8394
Macro-F1 gain: +0.0446

CORRECTION SAFETY
Evaluation records: 720
Corrections applied: 36
Beneficial: 32
Harmful: 3
Both labels wrong: 1
Net correct gain: 29
Beneficial fraction: 88.89%
Harmful fraction: 8.33%
Conflicts abstained: 101

STATISTICAL DIAGNOSTICS
Exact McNemar p-value: 4.17698e-07
Paired bootstrap 95% CI for accuracy gain (pp): [+2.5000, +5.6944]

PER-CLASS CORRECTIONS


,proposed_class,corrections,beneficial,harmful,both_wrong
0,World,0,0,0,0
1,Sports,5,5,0,0
2,Business,3,2,1,0
3,Sci/Tech,28,25,2,1



RESEARCH INTEGRITY
Frozen rule unchanged: YES
Evaluation decisions verified: YES
Development reference labels used: YES
Holdout reference labels accessed: NO
Training labels modified: NO
New LLM inference performed: NO
Report: NEW_SAVED
Saved path: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_calibration_v1/development_evaluation_report_v1.json

STATUS: FROZEN DEVELOPMENT EVALUATION COMPLETED


In [16]:

import json
import hashlib
from pathlib import Path

import pandas as pd

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
CAL_DIR = BASE / "outputs/s_slte_calibration_v1"

def audit_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def load_audit_json(filename):
    with (CAL_DIR / filename).open(encoding="utf-8") as f:
        return json.load(f)

split = load_audit_json("development_split_v1.json")
rule = load_audit_json("frozen_correction_rule_v1.json")
decision_manifest = load_audit_json(
    "evaluation_decision_manifest_v1.json"
)
report = load_audit_json(
    "development_evaluation_report_v1.json"
)

split_path = CAL_DIR / "development_split_v1.json"
rule_path = CAL_DIR / "frozen_correction_rule_v1.json"
decisions_path = CAL_DIR / "evaluation_decisions_v1.csv"

assert len(split["calibration_ids"]) == 1080
assert len(split["evaluation_ids"]) == 720
assert set(split["calibration_ids"]).isdisjoint(
    split["evaluation_ids"]
)

assert rule["min_neighbor_support"] == 0.80
assert rule["min_semantic_support_gap"] == 0.50
assert rule["k"] == 20

assert decision_manifest["rule_sha256"] == audit_sha256(rule_path)
assert decision_manifest["split_sha256"] == audit_sha256(split_path)
assert decision_manifest["decisions_csv_sha256"] == audit_sha256(
    decisions_path
)

assert report["rule_sha256"] == audit_sha256(rule_path)
assert report["split_sha256"] == audit_sha256(split_path)
assert report["decisions_sha256"] == audit_sha256(decisions_path)

decisions = pd.read_csv(decisions_path)

assert len(decisions) == 720
assert decisions["record_id"].is_unique
assert set(decisions["record_id"]) == set(
    split["evaluation_ids"]
)

assert int(decisions["correction_applied"].sum()) == 36
assert report["corrections_applied"] == 36

assert report["original_correct"] == 577
assert report["corrected_correct"] == 606
assert report["beneficial_corrections"] == 32
assert report["harmful_corrections"] == 3
assert report["both_labels_wrong"] == 1

assert report["net_correct_gain"] == 29
assert report["corrected_correct"] - report["original_correct"] == 29

assert sum(
    row["corrections"]
    for row in report["class_conditional_results"]
) == 36

assert report["evaluation_rule_modified"] is False
assert report["holdout_reference_labels_accessed"] is False
assert report["training_labels_modified"] is False

print("STAGE 12 — FINAL RESEARCH INTEGRITY CHECK")
print("Development annotations: 1800")
print("Calibration records: 1080")
print("Evaluation records: 720")
print("Applied corrections: 36")
print("Beneficial corrections: 32")
print("Harmful corrections: 3")
print("Net correct gain: +29")
print(
    "Accuracy gain:",
    f"{report['accuracy_gain_percentage_points']:+.4f} pp"
)
print(
    "Macro-F1 gain:",
    f"{report['macro_f1_gain']:+.4f}"
)
print("Frozen rule fingerprints: VERIFIED")
print("Decision artifact fingerprints: VERIFIED")
print("Holdout reference labels accessed: NO")
print("Training labels modified: NO")
print("New inference performed: NO")
print("\nSTATUS: STAGE 12 FINAL AUDIT PASSED")


STAGE 12 — FINAL RESEARCH INTEGRITY CHECK
Development annotations: 1800
Calibration records: 1080
Evaluation records: 720
Applied corrections: 36
Beneficial corrections: 32
Harmful corrections: 3
Net correct gain: +29
Accuracy gain: +4.0278 pp
Macro-F1 gain: +0.0446
Frozen rule fingerprints: VERIFIED
Decision artifact fingerprints: VERIFIED
Holdout reference labels accessed: NO
Training labels modified: NO
New inference performed: NO

STATUS: STAGE 12 FINAL AUDIT PASSED
